## Explore Bottom Boundary Layer Dynamics in Drygalksi Trough 2018 

#### Questions: 

1. What is the vertical structure of the bottom boundary layer at DITD and TNBD, are both moorings qualitatively similar? 

2. Are there periods that exhibit high shear and if so, can it be linked to certain phenomena 

3. What tidal periods dominate? 

4. Is there coherence between the TNBD and DITD? 

### New Outline 

Doesn't really make sense to look into thermal wind balance - would need cross trough observations to link to along trough - I don't have this. 

Instead, investigate rotating gravity current dynamics and see if we signs of cyclonic circulation hugging left side (hard to compare with right side of trough since we don't have measurements)

Steps: 

1. Gain a better understanding of mooring positions - what is distance between moorings and how far away in terms of km E,N,W,S

2. Make timeseries plots of nonrotated coordinates first 

3. Figure out how to rotate velocity components to be along-trough and across-trough (rotation matrix, need to find angle of trough relative to North or East) 

4. With rotated velocity (and should also do with non-rotated to compare) estimate shear at each mooring, max speeds, current direction (windrose plots and quiver timeseries)

5. Estimate brunt vaisalla frequnecy and see if it scales with shear - estimate richardson number, see if N^2 scales with speed and current direction 

6. Spectral analysis to estimate dominate tidal freqeuncies 

7. Low pass filter (or explicity remove frequency bands) to remove tides and do above analysis again 

8. Do analysis for data split into large N2 times (separate by using 90th percentile of PDF)

In [ ]:
# import packages 
import xarray as xr 
import numpy as np
import matplotlib.pyplot as plt 
import pandas as pd 
import gsw
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle

In [ ]:
plt.rcParams.update({
    "font.size": 12,          # base font size
    "axes.titlesize": 14,     # subplot titles
    "axes.labelsize": 11,     # x and y labels
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 16,
})


In [ ]:
%matplotlib widget

In [ ]:
def Calculate_TEOS10_vars(ds):

    if 'sal' not in ds:     # this skips D/2017/ctd which is not a dataset 
        return ds
    
    ds = ds.copy()  # important: DatasetView → writable Dataset

    ds['abs_sal'] = (
        'time',
        gsw.SA_from_SP(
            SP=ds['sal'].values,
            p=ds['z'].values,
            lon=ds['lon'].values,
            lat=ds['lat'].values,
        )
    )

    ds['theta'] = (
        'time',
        gsw.pt_from_t(
            SA=ds['abs_sal'].values,
            t=ds['temp'].values,
            p=ds['z'].values,
            p_ref=0,
        )
    )

    ds['CT'] = (
        'time',
        gsw.CT_from_pt(
            SA=ds['abs_sal'].values,
            pt=ds['theta'].values,
        )
    )

    ds['sigma0'] = (
        'time',
        gsw.sigma0(
            SA=ds['abs_sal'].values,
            CT=ds['CT'].values,
        )
    )

    return ds

import numpy as np

def calculate_direction(east,north):
    """
    This function calculates direction given east and north velocity components 

    ------
    Parameters 
    ------
    Inputs:
        - east: east velocity component; array 
        - north: north velocity component; array

    Outputs:
        - direction: calculated direction [degrees]; array 
    """
    
    direction = np.mod(np.arctan2(east,north)*180/np.pi,360)

    return direction 
    

def calculate_true_direction(direction,magnetic_declination):
    """
    Function that calculates true direction given a direction array in degrees and magnetic declination
    in degrees 

    -----
    Parameters
    -----

    Inputs: 
        - direction: array; direction in degrees 
        - magnetic_declination: float; magnetic declination angle in degrees 

    Outputs: 
        - true_direction: array; true direction in degrees (i.e. direction accounted for magnetic 
        declination) 
    
    """

    true_direction = np.mod(direction+magnetic_declination,360)

    return true_direction 
    
def rotate_velocity(u,v,angle):
    """Function that rotates velocity component vectors using rotation equations
       
       Inputs: v: u velocity, v velocity, angle: angle between vectors [radians]
       Outputs: rotated v velocity vectors 
    """
    u_prime = u*np.cos(angle) - v*np.sin(angle)
    v_prime = u*np.sin(angle) + v*np.cos(angle)
    
    return u_prime,v_prime



### Import and format data 

In [ ]:
# load in data from MAC 
# DITD
DITD_ctd_top = xr.open_dataset('DITD_2018_ctd_top.nc')
DITD_ctd_bot = xr.open_dataset('DITD_2018_ctd_bot.nc')
DITD_vel_top = xr.open_dataset('vel_1100.nc')
DITD_vel_bot = xr.open_dataset('vel_1200.nc')

# TNBD 
TNBD_ctd_bot = xr.open_dataset('tnbd_2018_ctd_bot_1093.nc')
TNBD_ctd_top = xr.open_dataset('tnbd_2018_ctd_top_999.nc')
TNBD_vel_bot = xr.open_dataset('tnbd_2018_vel_bot_1093.nc')
TNBD_vel_top = xr.open_dataset('tnbd_2018_vel_top_999.nc')



In [ ]:
# load in moorings datatree for mooring D data 
moorings = xr.open_datatree('/Users/mackenzie/Documents/Research/PhD/TNB_tools/preprocess/TNB_Moorings_DataTree.nc')

In [ ]:
D_vel_bot = moorings['D/2017/vel/bot'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))
D_vel_mid = moorings['D/2017/vel/mid'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))
D_ctd_mid = moorings['D/2017/ctd/mid'].to_dataset().sel(time=slice('2018-03-01','2019-01-01'))

In [ ]:
D_ctd_mid = Calculate_TEOS10_vars(D_ctd_mid)

In [ ]:
ldeo = moorings['LDEO/2017/ctd/bot'].to_dataset()

In [ ]:
# load in relevant data - TNBD and DITD ON DELL 
# DITD_ctd_top = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/DITD_2018_ctd_top.nc')
# DITD_ctd_bot = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/DITD_2018_ctd_bot.nc')
# DITD velocity data (NOTE TIME IS NOT CORRECTED IN THIS SAVED DATA  - NEED TO CORRECT AND RE-SAVE)
# DITD_vel_top = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/vel_1100.nc')
# DITD_vel_bot = xr.open_dataset('/Users/mblan/Documents/TNB_Research/data/DITD_2018_2019/vel_1200.nc')

In [ ]:
# fix DITD time - should save this change permenantly later
DITD_vel_top['time'] = DITD_vel_top.time - pd.Timedelta(12,'h')

In [ ]:
# rename variables in DITD velocity dataset 
DITD_vel_top = DITD_vel_top.rename({
    'east':'u',
    'north':'v',
    'up':'w',
    'pressure':'pres',
    'direction':'dir'
}).assign_coords({
    'z':1121,
    'lat':-75.28,
    'lon':164.1
})
# rename variables in DITD velocity dataset 
DITD_vel_bot = DITD_vel_bot.rename({
    'east':'u',
    'north':'v',
    'up':'w',
    'pressure':'pres',
    'direction':'dir'
}).assign_coords({
    'z':1218,
    'lat':-75.28,
    'lon':164.1
})

In [ ]:
# convert velocity from m/s to cm/s 
DITD_vel_top['u'] = DITD_vel_top.u * 100
DITD_vel_top['v'] = DITD_vel_top.v * 100
DITD_vel_top['speed'] = DITD_vel_top.speed * 100

DITD_vel_bot['v'] = DITD_vel_bot.v * 100
DITD_vel_bot['u'] = DITD_vel_bot.u * 100
DITD_vel_bot['speed'] = DITD_vel_bot.speed * 100


In [ ]:
# crop DITD vel bot to remove bad time at end of timeseries
DITD_vel_bot = DITD_vel_bot.sel(time=slice(None,'2018-06-21T20:00'))

In [ ]:
# # loadi n TNBD data 
# TNBD_ctd_bot = xr.open_dataset('//thepenguin/penguin2/Data/TNB/mooring/LDEO/TNBD_2018/working/tnbd_2018_ctd_bot_1093.nc')
# TNBD_ctd_top = xr.open_dataset('//thepenguin/penguin2/Data/TNB/mooring/LDEO/TNBD_2018/working/tnbd_2018_ctd_top_999.nc')
# TNBD_vel_bot = xr.open_dataset('//thepenguin/penguin2/Data/TNB/mooring/LDEO/TNBD_2018/working/tnbd_2018_vel_bot_1093.nc')
# TNBD_vel_top = xr.open_dataset('//thepenguin/penguin2/Data/TNB/mooring/LDEO/TNBD_2018/working/tnbd_2018_vel_top_999.nc')

In [ ]:
# to save some memory 
del moorings

Question: When defining depth of mooring should this be based off mean pressure

### Time Series Plots 

#### Examine hydrography and velocity. What are the difference between moorings? 

note: Mooring D velocity shows shift / time lag in 2018. Started in Dec. 2017, need to figure out if this is real signal. No evidence of time getting messed up 

Note: For mooring D: using CTD data from mid depth (722 meters) and velocity data from 1135 meters and 911 meters so almost 300 m spacing between velocity senors (need to keep this in mind when interpreting results, compared with TNBD and DITD that are ~100 meters apart and are paired wiht CTD at same depths )

In [ ]:
# time series analysis 
fig = plt.figure(figsize=(10,8),constrained_layout=True)

# define 3 hr rolling averages 
ds1 = TNBD_vel_top.rolling(time=6).mean()
ds2 = TNBD_vel_bot.rolling(time=12).mean()
ds3 = TNBD_ctd_top.rolling(time=180).mean()
ds4 = TNBD_ctd_bot.rolling(time=60).mean()

# Plot 1: T/S 
ax1 = plt.subplot(511)
plt.plot(ds3.time,ds3.sal,label='1031 m',lw=1,color='tab:blue')
plt.plot(ds4.time,ds4.sal,label='1128 m',color='tab:orange',linewidth=1)
plt.ylabel('Practical \n Salinity')
# ax.set_ylim(34.75,34.88)
ax1.set_xticks([])
plt.legend()
ax1.text(0.95, 0.89, 'A.', transform=ax1.transAxes, fontsize=10, fontweight='bold')

# Plot 2: Temp 
ax2 = plt.subplot(512)
plt.plot(ds3.time,ds3.temp,label='999m',lw=1,color='tab:blue')
plt.plot(ds4.time,ds4.temp,label='1100m',color='tab:orange',linewidth=1)
plt.ylabel('In-Situ \n Temperature \n ($^{\circ}$C)')
plt.xticks([])
ax2.text(0.95, 0.89, 'B.', transform=ax2.transAxes, fontsize=10, fontweight='bold')

# Plot 3: Potential Density  
ax3 = plt.subplot(513)
plt.plot(ds3.time,ds3.sigma0,label='999m',lw=1,color='tab:blue')
plt.plot(ds4.time,ds4.sigma0,label='1100m',color='tab:orange',linewidth=1)
plt.ylabel('Potential \n Density (kg/m$^3$)')
plt.xticks([])
ax3.text(0.95, 0.89, 'C.', transform=ax3.transAxes, fontsize=10, fontweight='bold')

# Plot 4: speed
ax4 = plt.subplot(514)
plt.plot(ds2.time,ds2.speed,label='1100m',color='tab:orange',linewidth=1)
plt.plot(ds1.time,ds1.speed,label='999m',lw=1,color='tab:blue')
plt.ylabel('Speed (cm/s)')
plt.xticks([])
ax4.text(0.95, 0.89, 'D.', transform=ax4.transAxes, fontsize=10, fontweight='bold')

# Plot 5: Quiver Plot (Direction)
ax5 = plt.subplot(515)
resample = 'W'
q = plt.quiver(
    ds1.time.resample(time=resample).mean().values,np.zeros(len(ds1.time.resample(time=resample).mean())),
    ds1.u.resample(time=resample).mean(),
    ds1.v.resample(time=resample).mean(),
    scale=80, units='width',color='tab:blue',label='1031 m'
)
q = plt.quiver(
    ds2.time.resample(time=resample).mean().values,np.zeros(len(ds2.time.resample(time=resample).mean())),
    ds2.u.resample(time=resample).mean(),
    ds2.v.resample(time=resample).mean(),
    scale=80, units='width',color='tab:orange',label='1128 m'
)
plt.quiverkey(q, X=0.85, Y=0.1, U=5, label='5 cm/s', labelpos='E')
# plt.ylabel('Speed (m/s)')
plt.yticks([])
plt.legend(loc='upper left')
plt.xticks(rotation=45)
plt.xlabel('Time (Year-Month)')
ax5.text(0.95, 0.89, 'E.', transform=ax5.transAxes, fontsize=10, fontweight='bold')

# plt.savefig('TNBD_TimeSeries_2018.png',dpi=600, bbox_inches="tight")

In [ ]:
# time series analysis 
fig = plt.figure(figsize=(10,8),constrained_layout=True)

# define 3 hr rolling averages 
ds1 = DITD_vel_top.rolling(time=18).mean()
ds2 = DITD_vel_bot.rolling(time=18).mean()
ds3 = DITD_ctd_top.rolling(time=180).mean()
ds4 = DITD_ctd_bot.rolling(time=60).mean()

# Plot 1: T/S 
ax1 = plt.subplot(511)
plt.plot(ds3.time,ds3.sal,label='1121 m',lw=1,color='tab:blue')
plt.plot(ds4.time,ds4.sal,label='1218 m',color='tab:orange',lw=1)
plt.ylabel('Practical \n Salinity')
# ax.set_ylim(34.75,34.88)
ax1.set_xticks([])
plt.legend()
ax1.text(0.95, 0.89, 'A.', transform=ax1.transAxes, fontsize=10, fontweight='bold')

# Plot 2: Temp 
ax2 = plt.subplot(512)
plt.plot(ds3.time,ds3.temp,label='999m',lw=1,color='tab:blue')
plt.plot(ds4.time,ds4.temp,label='1100m',color='tab:orange',lw=1)
plt.ylabel('In-Situ \n Temperature \n ($^{\circ}$C)')
plt.xticks([])
ax2.text(0.95, 0.89, 'B.', transform=ax2.transAxes, fontsize=10, fontweight='bold')

# Plot 3: Potential Density  
ax3 = plt.subplot(513)
plt.plot(ds3.time,ds3.sigma0-1000,label='999m',lw=1,color='tab:blue')
plt.plot(ds4.time,ds4.sigma0-1000,label='1100m',color='tab:orange',lw=1)
plt.ylabel('Potential \n Density (kg/m$^3$)')
plt.xticks([])
ax3.text(0.95, 0.89, 'C.', transform=ax3.transAxes, fontsize=10, fontweight='bold')

# Plot 4: speed
ax4 = plt.subplot(514)
plt.plot(ds2.time,ds2.speed,label='1100m',color='tab:orange',lw=1)
plt.plot(ds1.time,ds1.speed,label='999m',lw=1,color='tab:blue')
plt.ylabel('Speed (cm/s)')
plt.xticks([])
ax4.text(0.95, 0.89, 'D.', transform=ax4.transAxes, fontsize=10, fontweight='bold')

# Plot 5: Quiver Plot (Direction)
ax5 = plt.subplot(515)
resample = 'W'
q = plt.quiver(
    ds1.time.resample(time=resample).mean().values,np.zeros(len(ds1.time.resample(time=resample).mean())),
    ds1.u.resample(time=resample).mean(),
    ds1.v.resample(time=resample).mean(),
    scale=80, units='width',color='tab:blue',label='1121 m'
)
q = plt.quiver(
    ds2.time.resample(time=resample).mean().values,np.zeros(len(ds2.time.resample(time=resample).mean())),
    ds2.u.resample(time=resample).mean(),
    ds2.v.resample(time=resample).mean(),
    scale=80, units='width',color='tab:orange',label='1218 m'
)
plt.quiverkey(q, X=0.85, Y=0.1, U=5, label='5 cm/s', labelpos='E')
# plt.ylabel('Speed (m/s)')
plt.yticks([])
plt.legend(loc='lower left')
plt.xticks(rotation=45)
plt.xlabel('Time (Year-Month)')
ax5.text(0.95, 0.89, 'E.', transform=ax5.transAxes, fontsize=10, fontweight='bold')

# plt.savefig('DITD_TimeSeries_2018.png',dpi=600, bbox_inches="tight")

In [ ]:
# time series analysis 
fig = plt.figure(figsize=(10,8),constrained_layout=True)

# define 3 hr rolling averages 
ds1 = D_vel_mid.rolling(time=6).mean()
ds2 = D_vel_bot.rolling(time=6).mean()
ds3 = D_ctd_mid.rolling(time=6).mean()
# ds4 = DITD_ctd_bot.rolling(time=60).mean()

# Plot 1: T/S 
ax1 = plt.subplot(511)
plt.plot(ds3.time,ds3.sal,label='772 m',lw=1,color='tab:blue')
# plt.plot(ds4.time,ds4.sal,label='1218 m',color='tab:orange',lw=1)
plt.ylabel('Practical \n Salinity')
# ax.set_ylim(34.75,34.88)
ax1.set_xticks([])
plt.legend()
ax1.text(0.95, 0.89, 'A.', transform=ax1.transAxes, fontsize=10, fontweight='bold')

# Plot 2: Temp 
ax2 = plt.subplot(512)
plt.plot(ds3.time,ds3.temp,label='999m',lw=1,color='tab:blue')
# plt.plot(ds4.time,ds4.temp,label='1100m',color='tab:orange',lw=1)
plt.ylabel('In-Situ \n Temperature \n ($^{\circ}$C)')
plt.xticks([])
ax2.text(0.95, 0.89, 'B.', transform=ax2.transAxes, fontsize=10, fontweight='bold')

# Plot 3: Potential Density  
ax3 = plt.subplot(513)
plt.plot(ds3.time,ds3.sigma0,label='999m',lw=1,color='tab:blue')
# plt.plot(ds4.time,ds4.sigma0-1000,label='1100m',color='tab:orange',lw=1)
plt.ylabel('Potential \n Density (kg/m$^3$)')
plt.xticks([])
ax3.text(0.95, 0.89, 'C.', transform=ax3.transAxes, fontsize=10, fontweight='bold')

# Plot 4: speed
ax4 = plt.subplot(514)
plt.plot(ds2.time,(np.sqrt(ds2.u**2 + ds2.v**2)),label='1135m',color='tab:orange',lw=1)
plt.plot(ds1.time,(np.sqrt(ds1.u**2 + ds1.v**2)),label='911m',lw=1,color='tab:blue')
plt.ylabel('Speed (cm/s)')
plt.xticks([])
ax4.text(0.95, 0.89, 'D.', transform=ax4.transAxes, fontsize=10, fontweight='bold')
plt.legend(loc='upper left')

# Plot 5: Quiver Plot (Direction)
ax5 = plt.subplot(515)
resample = 'W'
q = plt.quiver(
    ds1.time.resample(time=resample).mean().values,np.zeros(len(ds1.time.resample(time=resample).mean())),
    ds1.u.resample(time=resample).mean(),
    ds1.v.resample(time=resample).mean(),
    scale=100, units='width',color='tab:blue',label='911 m'
)
q = plt.quiver(
    ds2.time.resample(time=resample).mean().values,np.zeros(len(ds2.time.resample(time=resample).mean())),
    ds2.u.resample(time=resample).mean(),
    ds2.v.resample(time=resample).mean(),
    scale=150, units='width',color='tab:orange',label='1135 m'
)
plt.quiverkey(q, X=0.85, Y=0.1, U=5, label='5 cm/s', labelpos='E')
# plt.ylabel('Speed (m/s)')
plt.yticks([])
plt.legend(loc='lower left')
plt.xticks(rotation=45)
plt.xlabel('Time (Year-Month)')
ax5.text(0.95, 0.89, 'E.', transform=ax5.transAxes, fontsize=10, fontweight='bold')

# plt.savefig('D_TimeSeries_2018.png',dpi=600, bbox_inches="tight")

In [ ]:
# T / S time series plots for DITD, D, TNBD 
# define time range 
time_range = slice('2018-03-14','2018-12-31')
fig = plt.figure(figsize=(12,8.5),constrained_layout=True)

ax1 = plt.subplot(321)
ds3 = TNBD_ctd_top.rolling(time=180).mean().sel(time=time_range)
ds4 = TNBD_ctd_bot.rolling(time=60).mean().sel(time=time_range)
plt.plot(ds3.time,ds3.sal,label='1031 m',lw=1,color='tab:blue')
plt.plot(ds4.time,ds4.sal,label='1128 m',color='tab:orange',linewidth=1)
plt.ylabel('Practical \n Salinity')
ax1.tick_params(axis='x', labelbottom=False)
plt.legend()
ax1.text(0.95, 0.89, 'A.', transform=ax1.transAxes, fontsize=10, fontweight='bold')
ax1.text(0.9, 0.05, 'TNBD', transform=ax1.transAxes, fontsize=10, fontweight='bold')
plt.ylim(34.765,34.875)

# Plot 2: Temp 
ax2 = plt.subplot(322)
plt.plot(ds3.time,ds3.theta,label='999m',lw=1,color='tab:blue')
plt.plot(ds4.time,ds4.theta,label='1100m',color='tab:orange',linewidth=1)
plt.ylabel('Potential \n Temperature ($^{\circ}$C)')
ax2.tick_params(axis='x', labelbottom=False)
ax2.text(0.95, 0.89, 'B.', transform=ax2.transAxes, fontsize=10, fontweight='bold')
ax2.text(0.9, 0.05, 'TNBD', transform=ax2.transAxes, fontsize=10, fontweight='bold')
plt.ylim(-1.934,-1.910)


# Plot 1: T/S 
ax3 = plt.subplot(323)
ds3 = D_ctd_mid.rolling(time=6).mean().sel(time=time_range)
plt.plot(ds3.time,ds3.sal,label='772 m',lw=1,color='tab:blue')
# plt.plot(ds4.time,ds4.sal,label='1218 m',color='tab:orange',lw=1)
plt.ylabel('Practical \n Salinity')
# ax.set_ylim(34.75,34.88)
# plt.xticks(rotation=45)
ax3.tick_params(axis='x', labelbottom=False)
plt.legend()
ax3.text(0.95, 0.89, 'C.', transform=ax3.transAxes, fontsize=10, fontweight='bold')
ax3.text(0.83, 0.05, 'Mooring D', transform=ax3.transAxes, fontsize=10, fontweight='bold')
plt.ylim(34.765,34.875)


# Plot 2: Temp 
ax4 = plt.subplot(324)
plt.plot(ds3.time,ds3.theta,label='999m',lw=1,color='tab:blue')
# plt.plot(ds4.time,ds4.temp,label='1100m',color='tab:orange',lw=1)
plt.ylabel('Potential \n Temperature ($^{\circ}$C)')
ax4.tick_params(axis='x', labelbottom=False)
ax4.text(0.95, 0.89, 'D.', transform=ax4.transAxes, fontsize=10, fontweight='bold')
ax4.text(0.83, 0.05, 'Mooring D', transform=ax4.transAxes, fontsize=10, fontweight='bold')
plt.ylim(-1.934,-1.910)


# Plot 1: T/S 
ax5 = plt.subplot(325)
ds3 = DITD_ctd_top.rolling(time=180).mean().sel(time=time_range)
ds4 = DITD_ctd_bot.rolling(time=60).mean().sel(time=time_range)
plt.plot(ds3.time,ds3.sal,label='1121 m',lw=1,color='tab:blue')
plt.plot(ds4.time,ds4.sal,label='1218 m',color='tab:orange',lw=1)
plt.ylabel('Practical \n Salinity')
# ax.set_ylim(34.75,34.88)
# ax5.tick_params(axis='x', labelbottom=False)
plt.legend()
ax5.text(0.95, 0.89, 'E.', transform=ax5.transAxes, fontsize=10, fontweight='bold')
ax5.text(0.9, 0.05, 'DITD', transform=ax5.transAxes, fontsize=10, fontweight='bold')
plt.ylim(34.765,34.875)
# Major ticks = every month
ax5.xaxis.set_major_locator(mdates.MonthLocator())

# Format tick labels as abbreviated month names
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax5.set_xlabel('2018')

# Plot 2: Temp 
ax6 = plt.subplot(326)
plt.plot(ds3.time,ds3.theta,label='999m',lw=1,color='tab:blue')
plt.plot(ds4.time,ds4.theta,label='1100m',color='tab:orange',lw=1)
plt.ylabel('Potential \n Temperature ($^{\circ}$C)')
# ax6.tick_params(axis='x', labelbottom=False)
ax6.text(0.95, 0.89, 'F.', transform=ax6.transAxes, fontsize=10, fontweight='bold')
ax6.text(0.9, 0.05, 'DITD', transform=ax6.transAxes, fontsize=10, fontweight='bold')
plt.ylim(-1.934,-1.910)
# Major ticks = every month
ax6.xaxis.set_major_locator(mdates.MonthLocator())

# Format tick labels as abbreviated month names
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax6.set_xlabel('2018')
# plt.savefig('2018_Temp_sal_quals.png',dpi=600, bbox_inches="tight")

In [ ]:
fig = plt.figure(figsize=(12,8.5),constrained_layout=True)

ax1 = plt.subplot(321)
# define 3 hr rolling averages 
ds1 = TNBD_vel_top.rolling(time=6).mean().sel(time=time_range)
ds2 = TNBD_vel_bot.rolling(time=12).mean().sel(time=time_range)
plt.plot(ds2.time,ds2.speed,label='1100m',color='tab:orange',linewidth=1)
plt.plot(ds1.time,ds1.speed,label='999m',lw=1,color='tab:blue')
plt.ylabel('Speed (cm/s)')
ax1.tick_params(axis='x', labelbottom=False)
ax1.text(0.95, 0.89, 'A.', transform=ax1.transAxes, fontsize=10, fontweight='bold')
ax1.text(0.05, 0.89, 'TNBD', transform=ax1.transAxes, fontsize=10, fontweight='bold')
plt.ylim(0,40)

ax2 = plt.subplot(322)
resample = 'W'
q = plt.quiver(
    ds1.time.resample(time=resample).mean().values,np.zeros(len(ds1.time.resample(time=resample).mean())),
    ds1.u.resample(time=resample).mean(),
    ds1.v.resample(time=resample).mean(),
    scale=80, units='width',color='tab:blue',label='1031 m'
)
q = plt.quiver(
    ds2.time.resample(time=resample).mean().values,np.zeros(len(ds2.time.resample(time=resample).mean())),
    ds2.u.resample(time=resample).mean(),
    ds2.v.resample(time=resample).mean(),
    scale=80, units='width',color='tab:orange',label='1128 m'
)
plt.quiverkey(q, X=0.85, Y=0.1, U=5, label='5 cm/s', labelpos='E')
# plt.ylabel('Speed (m/s)')
plt.yticks([])
ax2.tick_params(axis='x', labelbottom=False)
plt.legend(loc='upper left')
# plt.xticks(rotation=45)
ax2.text(0.95, 0.89, 'B.', transform=ax2.transAxes, fontsize=10, fontweight='bold')
ax2.text(0.21, 0.89, 'TNBD', transform=ax2.transAxes, fontsize=10, fontweight='bold')
ax2.axhline(0, color='k', linewidth=1.5,alpha=0.7, linestyle='dashed')

ax3 = plt.subplot(323)
# define 3 hr rolling averages 
ds1 = D_vel_mid.rolling(time=6).mean()
ds2 = D_vel_bot.rolling(time=6).mean()
plt.plot(ds2.time,(np.sqrt(ds2.u**2 + ds2.v**2)),label='1135m',color='tab:orange',lw=1)
plt.plot(ds1.time,(np.sqrt(ds1.u**2 + ds1.v**2)),label='911m',lw=1,color='tab:blue')
plt.ylabel('Speed (cm/s)')
plt.ylim(0,40)
ax3.tick_params(axis='x', labelbottom=False)
ax3.text(0.95, 0.89, 'C.', transform=ax3.transAxes, fontsize=10, fontweight='bold')
ax3.text(0.03, 0.89, 'Mooring D', transform=ax3.transAxes, fontsize=10, fontweight='bold')
# plt.legend(loc='upper left')

# Plot 5: Quiver Plot (Direction)
ax4 = plt.subplot(324)
resample = 'W'
q = plt.quiver(
    ds1.time.resample(time=resample).mean().values,np.zeros(len(ds1.time.resample(time=resample).mean())),
    ds1.u.resample(time=resample).mean(),
    ds1.v.resample(time=resample).mean(),
    scale=80, units='width',color='tab:blue',label='911 m'
)
q = plt.quiver(
    ds2.time.resample(time=resample).mean().values,np.zeros(len(ds2.time.resample(time=resample).mean())),
    ds2.u.resample(time=resample).mean(),
    ds2.v.resample(time=resample).mean(),
    scale=80, units='width',color='tab:orange',label='1135 m'
)
plt.quiverkey(q, X=0.85, Y=0.1, U=5, label='5 cm/s', labelpos='E')
# plt.ylabel('Speed (m/s)')
plt.yticks([])
plt.legend(loc='lower left')
ax4.tick_params(axis='x', labelbottom=False)
ax4.text(0.95, 0.89, 'D.', transform=ax4.transAxes, fontsize=10, fontweight='bold')
ax4.text(0.03, 0.89, 'Mooring D', transform=ax4.transAxes, fontsize=10, fontweight='bold')





ax5 = plt.subplot(325)
# define 3 hr rolling averages 
ds1 = DITD_vel_top.rolling(time=18).mean().sel(time=time_range)
ds2 = DITD_vel_bot.rolling(time=18).mean().sel(time=time_range)
plt.plot(ds2.time,ds2.speed,label='1100m',color='tab:orange',lw=1)
plt.plot(ds1.time,ds1.speed,label='999m',lw=1,color='tab:blue')
plt.ylabel('Speed (cm/s)')
# ax5.tick_params(axis='x', labelbottom=False)
ax5.text(0.95, 0.89, 'E.', transform=ax5.transAxes, fontsize=10, fontweight='bold')
ax5.text(0.05, 0.89, 'DITD', transform=ax5.transAxes, fontsize=10, fontweight='bold')
# Major ticks = every month
ax5.xaxis.set_major_locator(mdates.MonthLocator())


# Format tick labels as abbreviated month names
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax5.set_xlabel('2018')

plt.ylim(0,40)

# Plot 5: Quiver Plot (Direction)
ax6 = plt.subplot(326)
resample = 'W'
q = plt.quiver(
    ds1.time.resample(time=resample).mean().values,np.zeros(len(ds1.time.resample(time=resample).mean())),
    ds1.u.resample(time=resample).mean(),
    ds1.v.resample(time=resample).mean(),
    scale=80, units='width',color='tab:blue',label='1121 m'
)
q = plt.quiver(
    ds2.time.resample(time=resample).mean().values,np.zeros(len(ds2.time.resample(time=resample).mean())),
    ds2.u.resample(time=resample).mean(),
    ds2.v.resample(time=resample).mean(),
    scale=80, units='width',color='tab:orange',label='1218 m'
)
plt.quiverkey(q, X=0.85, Y=0.1, U=5, label='5 cm/s', labelpos='E')
# plt.ylabel('Speed (m/s)')
plt.yticks([])
plt.legend(loc='lower left')
# ax6.tick_params(axis='x', labelbottom=False)
ax6.text(0.95, 0.89, 'F.', transform=ax6.transAxes, fontsize=10, fontweight='bold')
ax6.text(0.05, 0.89, 'DITD', transform=ax6.transAxes, fontsize=10, fontweight='bold')
ax6.axhline(0, color='k', linewidth=1.5,alpha=0.7, linestyle='dashed')

# Major ticks = every month
ax6.xaxis.set_major_locator(mdates.MonthLocator())
ax6.axhline(0, color='k', linewidth=1.5,alpha=0.7, linestyle='dashed')

# Format tick labels as abbreviated month names
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax6.set_xlabel('2018')

# plt.savefig('2018_speed_currentdirection_quals.png',dpi=600, bbox_inches="tight")


In [ ]:
plt.close('all')

Things to note in paper / presentation: 

1. Velocity magnitude, similarity in magnitude with depth, similar current direction at both depths for DITD and D, lots of variation in current direction at TNBD 

2. Calculate salinity increase during HSSW arrival (maybe put this in HSSW arrival section)

3. Density is fully dominated by salintiy 

4. HSSW arrival paired with decrease in temperature at TNBD and DITD, not a big decrease in temp at D 

5. Cool & fresh event around June for Mooring D, perhaps seen later in July at DITD and TNBD ( would take a longer time to mix down becuase it is lighter )


### T/S Diagram 

Idea: Make T/S diagram showing all mooring data available in 2018 to show water masses in Terra Nova Bay / Shelf, zoomed in panel showing HSSW at D, DITD, and TNBD (this zoomed in section will be focus of the paper)

In [ ]:
def TS_template(ax1):
    """ 
    """
    # Create figure
    # fig = plt.figure(figsize=(10, 6),constrained_layout=True)
    # # Top left square
    # ax1 = plt.subplot(1,1,1)

    mint=-2.3
    maxt=-1.4
    mins=34.4
    maxs=35

    tempL=np.linspace(mint,maxt)
    salL=np.linspace(mins,maxs)
    Tg, Sg = np.meshgrid(tempL,salL)

    lon = 165
    lat = -75.0112
    p = 1093
    SA = gsw.SA_from_SP(Sg, p, lon, lat)
    CT = gsw.CT_from_pt(SA, Tg)
    # rho = gsw.rho(SA, Tg, 0)
    # sigma_theta = rho - 1000

    # CALCULATE DENSITY (METHOD 2)
    sigma_theta = gsw.sigma0(SA, CT)

    t_freezing1100 = gsw.t_freezing(gsw.SA_from_SP(salL, 0, lon, lat), p=0, saturation_fraction=0)
    plt.plot(salL, t_freezing1100, 'k--', label='Freezing Line at 0 dbar')

    cs = ax1.contour(Sg, Tg, sigma_theta, colors='k', zorder=1, alpha=0.7, levels=4)
    cl=plt.clabel(cs,fontsize=10,inline=True,manual=[(34.45,-2.2),(34.55,-2.05),(34.75,-1.75)])



In [ ]:
fig = plt.figure(figsize=(8,8),constrained_layout=True)
# Top left square
ax1 = plt.subplot(1,1,1)
TS_template(ax1)

# Mooring D 
plt.scatter(D_ctd_mid.sal,D_ctd_mid.theta,label='D 772m',s=0.5)
# Mooring DITD
plt.scatter(DITD_ctd_top.sal,DITD_ctd_top.theta,label='DITD 1121m',s=0.5)
plt.scatter(DITD_ctd_bot.sal,DITD_ctd_bot.theta,label='DITD 1218m',s=0.5)
# Mooring TNBD 
plt.scatter(TNBD_ctd_top.sal,TNBD_ctd_top.theta,label='TNBD 1031m',s=0.5)
plt.scatter(TNBD_ctd_bot.sal,TNBD_ctd_bot.theta,label='TNBD 1128',s=0.5)
# Mooring LDEO 
# plt.scatter(ldeo.sal,ldeo.theta,label='LDEO',s=0.5)
leg = plt.legend(markerscale=4)
leg.set_zorder(11)
# plt.text(34.7,-2,"TISW",fontweight='bold')
# plt.text(34.8,-1.95,"HSSW",fontweight='bold')
# plt.text(34.58,-1.8,"AASW",fontweight='bold')
plt.xlim(34.7,34.9)
plt.ylim(-2,-1.8)

# # --- Define water mass boxes (salinity_min, theta_min, width, height)

# # TISW
# tisw_box = Rectangle((34.68, -2.15),   # lower-left corner
#                      0.08,             # salinity width
#                      0.20,             # theta height
#                      facecolor='lightblue',
#                      alpha=0.4,
#                      edgecolor='grey',
#                      zorder=0)

# # HSSW
# hssw_box = Rectangle((34.78, -2.05),
#                      0.12,
#                      0.15,
#                      facecolor='lightcoral',
#                      alpha=0.4,
#                      edgecolor='grey',
#                      zorder=0)

# # AASW
# aasw_box = Rectangle((34.50, -1.90),
#                      0.12,
#                      0.15,
#                      facecolor='lightgreen',
#                      alpha=0.4,
#                      edgecolor='grey',
#                      zorder=0)

# # Add to axis
# ax1.add_patch(tisw_box)
# ax1.add_patch(hssw_box)
# ax1.add_patch(aasw_box)

In [ ]:
import cmocean

In [ ]:
# make subplots of TS diagrams colored in by time at each mooring 
fig = plt.figure(figsize=(10,11),constrained_layout=True)
# define some things 
# ylim 
ymin = -1.94
ymax = -1.9
xmin = 34.75
xmax = 34.9 
# Top left square
ax1 = plt.subplot(321)
TS_template(ax1)
# Mooring TNBD 
# Convert time to matplotlib numeric format
time_num = mdates.date2num(TNBD_ctd_top.time.values)
vmin = time_num.min()
vmax = time_num.max()
ticks = np.linspace(vmin, vmax, 6)
sc = plt.scatter(TNBD_ctd_top.sal,TNBD_ctd_top.theta,c=time_num, cmap = cmocean.cm.thermal,label='TNBD 1031m',s=0.5)
plt.xlim(xmin,xmax)
plt.ylim(ymin,ymax)
plt.xticks(rotation=45)
cbar = plt.colorbar(sc,ticks=ticks)
# cbar.set_label("Time")
cbar.ax.yaxis_date()
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.ylabel('Potential Temperature $^{\circ}$C')
ax1.text(0.95, 0.93, 'A.', transform=ax1.transAxes, fontsize=8, fontweight='bold')
ax1.text(0.01, 0.93, 'TNBD 1031m', transform=ax1.transAxes, fontsize=8, fontweight='bold')

ax2 = plt.subplot(322)
TS_template(ax2)
time_num = mdates.date2num(TNBD_ctd_bot.time.values)
vmin = time_num.min()
vmax = time_num.max()
ticks = np.linspace(vmin, vmax, 6)
sc = plt.scatter(TNBD_ctd_bot.sal,TNBD_ctd_bot.theta,c=time_num, cmap = cmocean.cm.thermal,label='TNBD 1031m',s=0.5)
plt.xlim(xmin,xmax)
plt.ylim(ymin,ymax)
# plt.yticks([])
plt.xticks(rotation=45)
cbar = plt.colorbar(sc,ticks=ticks)
# cbar.set_label("Time")
cbar.ax.yaxis_date()
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax2.text(0.95, 0.93, 'B.', transform=ax2.transAxes, fontsize=8, fontweight='bold')
ax2.text(0.01, 0.93, 'TNBD 1128m', transform=ax2.transAxes, fontsize=8, fontweight='bold')

ax3 = plt.subplot(323)
TS_template(ax3)
time_num = mdates.date2num(DITD_ctd_top.time.values)
vmin = time_num.min()
vmax = time_num.max()
ticks = np.linspace(vmin, vmax, 6)
sc = plt.scatter(DITD_ctd_top.sal,DITD_ctd_top.theta,c=time_num, cmap = cmocean.cm.thermal,label='DITD 1031m',s=0.5)
plt.xlim(xmin,xmax)
plt.ylim(ymin,ymax)
plt.xticks(rotation=45)
cbar = plt.colorbar(sc,ticks=ticks)
# cbar.set_label("Time")
cbar.ax.yaxis_date()
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.ylabel('Potential Temperature $^{\circ}$C')
ax3.text(0.95, 0.93, 'C.', transform=ax3.transAxes, fontsize=8, fontweight='bold')
ax3.text(0.01, 0.93, 'DITD 1121m', transform=ax3.transAxes, fontsize=8, fontweight='bold')

ax4 = plt.subplot(324)
TS_template(ax4)
time_num = mdates.date2num(DITD_ctd_bot.time.values)
vmin = time_num.min()
vmax = time_num.max()
ticks = np.linspace(vmin, vmax, 6)
sc = plt.scatter(DITD_ctd_bot.sal,DITD_ctd_bot.theta,c=time_num, cmap = cmocean.cm.thermal,label='DITD 1031m',s=0.5)
plt.xlim(xmin,xmax)
plt.ylim(ymin,ymax)
# plt.yticks([])
cbar = plt.colorbar(sc,ticks=ticks)
# cbar.set_label("Time")
cbar.ax.yaxis_date()
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.xticks(rotation=45)
ax4.text(0.95, 0.93, 'D.', transform=ax4.transAxes, fontsize=8, fontweight='bold')
ax4.text(0.01, 0.93, 'DITD 1218m', transform=ax4.transAxes, fontsize=8, fontweight='bold')

ax5 = plt.subplot(325)
TS_template(ax5)
time_num = mdates.date2num(D_ctd_mid.time.values)
vmin = time_num.min()
vmax = time_num.max()
ticks = np.linspace(vmin, vmax, 6)
sc = plt.scatter(D_ctd_mid.sal,D_ctd_mid.theta,c=time_num, cmap = cmocean.cm.thermal,label='D 772m',s=0.5)
plt.xlim(xmin,xmax)
plt.ylim(ymin,ymax)
cbar = plt.colorbar(sc,ticks=ticks)
# cbar.set_label("Time")
cbar.ax.yaxis_date()
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter('%b'))
plt.ylabel('Potential Temperature $^{\circ}$C')
plt.xlabel('Practical Salinity')
plt.xticks(rotation=45)
ax5.text(0.95, 0.93, 'E.', transform=ax5.transAxes, fontsize=8, fontweight='bold')
ax5.text(0.01, 0.93, 'D 772m', transform=ax5.transAxes, fontsize=8, fontweight='bold')

ax6 = plt.subplot(326)
TS_template(ax6)
# Mooring D 
plt.scatter(D_ctd_mid.sal,D_ctd_mid.theta,s=0.5,color='k')
# Mooring DITD
plt.scatter(DITD_ctd_top.sal,DITD_ctd_top.theta,s=0.5,color='k')
plt.scatter(DITD_ctd_bot.sal,DITD_ctd_bot.theta,s=0.5,color='k')
# Mooring TNBD 
plt.scatter(TNBD_ctd_top.sal,TNBD_ctd_top.theta,s=0.5,color='k')
plt.scatter(TNBD_ctd_bot.sal,TNBD_ctd_bot.theta,s=0.5,color='k')
# Mooring LDEO 
# plt.scatter(ldeo.sal,ldeo.theta,label='LDEO',s=0.5)
leg = plt.legend(markerscale=2,fontsize=10,loc='upper left')
leg.set_zorder(11)
ax6.text(0.95, 0.93, 'F.', transform=ax6.transAxes, fontsize=8, fontweight='bold')
# ax6.text(0.95, 0.89, 'HSSW.', transform=ax6.transAxes, fontsize=10, fontweight='bold')
# plt.text(34.7,-2,"TISW",fontweight='bold')
# plt.text(34.8,-1.95,"HSSW",fontweight='bold')
# plt.text(34.58,-1.8,"AASW",fontweight='bold')
plt.xticks(rotation=45)
plt.xlabel('Practical Salinity')
# --- Define water mass boxes (salinity_min, theta_min, width, height)

# TISW
# tisw_box = Rectangle((34.68, -2.15),   # lower-left corner
#                      0.08,             # salinity width
#                      0.20,             # theta height
#                      facecolor='lightblue',
#                      alpha=0.4,
#                      edgecolor='grey',
#                      zorder=0)

# # HSSW
# hssw_box = Rectangle((34.78, -2.05),
#                      0.12,
#                      0.15,
#                      facecolor='lightcoral',
#                      alpha=0.4,
#                      edgecolor='grey',
#                      zorder=0)

# # AASW
# aasw_box = Rectangle((34.50, -1.90),
#                      0.12,
#                      0.15,
#                      facecolor='lightgreen',
#                      alpha=0.4,
#                      edgecolor='grey',
#                      zorder=0)

# # Add to axis
# ax6.add_patch(tisw_box)
# ax6.add_patch(hssw_box)
# ax6.add_patch(aasw_box)

# plt.savefig('TS_Diagram.png',dpi=600)

### Calculate Brunt Vaisalla Frequency, Shear, and Richardson Number 

Notes: 

- Can only calulate N^2 for DITD and TNBD 

- Calculate shear by sqrt(du/dz^2 + dv/dz^2)

- Richardson number is the ratio of N^2 to shear^2 (give hint at instabilities - Ri < 0.25 is unstable)

- Need to make sure sensors are aligned (i.e. resample to get same sampling frequency and select same time ranges) 

In [ ]:
# rotate coordinate system to be along and across trough 

# REVIST THIS LATER 

# theta_down = 35.49650756

# TNBD_vel_bot['u_along'] = ('time',(TNBD_vel_bot.u*np.cos(theta_down) + TNBD_vel_bot.v*np.sin(theta_down)).data)
# TNBD_vel_bot['u_cross'] = ('time',(-TNBD_vel_bot.u*np.sin(theta_down) + TNBD_vel_bot.v*np.cos(theta_down)).data)

# TNBD_vel_top['u_along'] = ('time',(TNBD_vel_top.u*np.cos(theta_down) + TNBD_vel_top.v*np.sin(theta_down)).data)
# TNBD_vel_top['u_cross'] = ('time',(-TNBD_vel_top.u*np.sin(theta_down) + TNBD_vel_top.v*np.cos(theta_down)).data)

# DITD_vel_top['u_along'] = ('time',(DITD_vel_top.u*np.cos(theta_down) + DITD_vel_top.v*np.sin(theta_down)).data)
# DITD_vel_top['u_cross'] = ('time',(-DITD_vel_top.u*np.sin(theta_down) + DITD_vel_top.v*np.cos(theta_down)).data)

# DITD_vel_bot['u_along'] = ('time',(DITD_vel_bot.u*np.cos(theta_down) + DITD_vel_bot.v*np.sin(theta_down)).data)
# DITD_vel_bot['u_cross'] = ('time',(-DITD_vel_bot.u*np.sin(theta_down) + DITD_vel_bot.v*np.cos(theta_down)).data)

# D_vel_mid['u_along'] = ('time',(D_vel_mid.u*np.cos(theta_down) + D_vel_mid.v*np.sin(theta_down)).data)
# D_vel_mid['u_cross'] = ('time',(-D_vel_mid.u*np.sin(theta_down) + D_vel_mid.v*np.cos(theta_down)).data)

# D_vel_bot['u_along'] = ('time',(D_vel_bot.u*np.cos(theta_down) + D_vel_bot.v*np.sin(theta_down)).data)
# D_vel_bot['u_cross'] = ('time',(-D_vel_bot.u*np.sin(theta_down) + D_vel_bot.v*np.cos(theta_down)).data)

In [ ]:
########### ALIGN TNBD ################
# data resolutions 
# ctd bottom - 3 min
# ctd top - 1 min
# vel bot - 15 min
# vel top - 30 min
# I will resample the other datasets to 30 min to do analysis between them 
TNBD_vel_bot = TNBD_vel_bot.resample(time='30min').mean().sel(time=slice('2018-03-14','2019-01-03T23:30'))
TNBD_ctd_bot = TNBD_ctd_bot.resample(time='30min').mean().sel(time=slice('2018-03-14','2019-01-03T23:30'))
TNBD_ctd_top = TNBD_ctd_top.resample(time='30min').mean().sel(time=slice('2018-03-14','2019-01-03T23:30'))

TNBD_vel_top = TNBD_vel_top.sel(time=slice('2018-03-14',None))

In [ ]:
########### ALIGN DITD ############## 

# resample data to same dt 
DITD_ctd_bot = DITD_ctd_bot.resample(time='30min').mean()
DITD_ctd_top = DITD_ctd_top.resample(time='30min').mean()
DITD_vel_bot = DITD_vel_bot.resample(time='30min').mean()
# align times 
DITD_vel_top = DITD_vel_top.resample(time='30min').mean()
DITD_ctd_bot = DITD_ctd_bot.sel(time=slice('2018-03-11',None))
DITD_ctd_top = DITD_ctd_top.sel(time=DITD_ctd_bot.time)

In [ ]:
############### ALIGN D ################## 
# D is already aligned, great! 

In [ ]:
# calculate shear at each mooring
### TNBD #### 
TNBD_dudz = (TNBD_vel_bot.u - TNBD_vel_top.u)/((TNBD_vel_bot.z - TNBD_vel_top.z)*100) # * 100 to account for m --> cm conversion! 
TNBD_dvdz = (TNBD_vel_bot.v - TNBD_vel_top.v)/((TNBD_vel_bot.z - TNBD_vel_top.z)*100)
TNBD_shear_mag = np.sqrt(TNBD_dudz**2 + TNBD_dvdz**2)

### DITD #### 
DITD_dudz = (DITD_vel_bot.u - DITD_vel_top.sel(time=DITD_vel_bot.time).u)/((DITD_vel_bot.z - DITD_vel_top.z)*100)
DITD_dvdz = (DITD_vel_bot.v - DITD_vel_top.sel(time=DITD_vel_bot.time).v)/((DITD_vel_bot.z - DITD_vel_top.z)*100)
DITD_shear_mag = np.sqrt(DITD_dudz**2 + DITD_dvdz**2)

### D #### 
D_dudz = (D_vel_bot.u.values - D_vel_mid.u.values)/((D_vel_bot.z.values - D_vel_mid.z.values)*100)
D_dvdz = (D_vel_bot.v.values - D_vel_mid.v.values)/((D_vel_bot.z.values - D_vel_mid.z.values)*100)
D_shear_mag = np.sqrt(D_dudz**2 + D_dvdz**2)

In [ ]:
# add shear to datasets 

TNBD_vel_bot['dudz'] = ('time', TNBD_dudz.data)
TNBD_vel_bot['dvdz'] = ('time', TNBD_dvdz.data)
TNBD_vel_bot['shear'] = ('time', TNBD_shear_mag.data)

DITD_vel_bot['dudz'] = ('time', DITD_dudz.data)
DITD_vel_bot['dvdz'] = ('time', DITD_dvdz.data)
DITD_vel_bot['shear'] = ('time', DITD_shear_mag.data)

D_vel_bot['dudz'] = ('time', D_dudz)
D_vel_bot['dvdz'] = ('time', D_dvdz)
D_vel_bot['shear'] = ('time', D_shear_mag)

In [ ]:
# Calculate N^2 for DITD and TNBD 
 
g = 9.81
rho_0 = 1028

TNBD_drhodz = (TNBD_ctd_bot.sigma0 - TNBD_ctd_top.sigma0) / (TNBD_vel_bot.z - TNBD_vel_top.z)
TNBD_N2 = (g/rho_0)*(TNBD_drhodz)

DITD_drhodz = (DITD_ctd_bot.sigma0 - DITD_ctd_top.sigma0) / (DITD_ctd_bot.z - DITD_ctd_top.z)
DITD_N2 = (g/rho_0)*(DITD_drhodz)



In [ ]:
# Add N2 to datasets
TNBD_ctd_bot['N2'] = ('time', TNBD_N2.data)
DITD_ctd_bot['N2'] = ('time', DITD_N2.data)

In [ ]:
# Calculate Richardson number for DITD and TNBD 

TNBD_Ri = TNBD_N2 / TNBD_shear_mag**2
DITD_Ri = DITD_N2 / DITD_shear_mag**2


In [ ]:
# plot the results 
fig = plt.figure(figsize=(11,8),constrained_layout=True)
time_range = slice('2018-03-14','2018-12-31')

ax = plt.subplot(311)
line1, = plt.plot(TNBD_vel_bot.sel(time=time_range).time,TNBD_vel_bot.sel(time=time_range).shear,label='Shear',zorder=1)
plt.ylabel('Shear (s$^{-1}$)',color='tab:blue')
# ax.text(0.95, 0.9, 'A.', transform=ax.transAxes, fontsize=10, fontweight='bold')
ax.text(0.02, 0.9, 'Mooring TNBD', transform=ax.transAxes, fontsize=10, fontweight='bold')
ax1 = ax.twinx()
line2, = ax1.plot(TNBD_ctd_bot.sel(time=time_range).time,TNBD_ctd_bot.sel(time=time_range).N2,zorder=10,color='tab:red',label='N$^2$')
ax1.set_ylabel('N$^2$ (s$^{-2}$)',color='tab:red')
ax.set_ylim(0,0.0042)
ax1.set_ylim(0,8.9e-6)
ax.tick_params(axis='x', labelrotation=45)
ax1.tick_params(axis='y',colors='tab:red')
ax.tick_params(axis='y', colors='tab:blue')
lines = [line1, line2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='best')
ax.tick_params(axis='x', labelbottom=False)

ax = plt.subplot(312)
plt.plot(DITD_vel_bot.sel(time=time_range).time,DITD_vel_bot.sel(time=time_range).shear,label='Shear',zorder=1)
plt.ylabel('Shear (s$^{-1}$)',color='tab:blue')
# ax.text(0.95, 0.9, 'B.', transform=ax.transAxes, fontsize=10, fontweight='bold')
ax.text(0.02, 0.9, 'Mooring DITD', transform=ax.transAxes, fontsize=10, fontweight='bold')
ax1 = ax.twinx()
ax1.plot(DITD_ctd_bot.sel(time=time_range).time,DITD_ctd_bot.sel(time=time_range).N2,zorder=10,color='tab:red')
ax1.set_ylabel('N$^2$ (s$^{-2}$)',color='tab:red')
ax.set_ylim(0,0.0042)
ax1.set_ylim(0,8.9e-6)
ax.tick_params(axis='x', labelrotation=45)
ax1.tick_params(axis='y',colors='tab:red')
ax.tick_params(axis='y', colors='tab:blue')
ax.tick_params(axis='x', labelbottom=False)

ax = plt.subplot(313)
line1, = plt.plot(D_vel_bot.sel(time=time_range).time,D_vel_bot.sel(time=time_range).shear,label='Shear',zorder=1)
plt.ylabel('Shear (s$^{-1}$)',color='tab:blue')
# ax.text(0.95, 0.9, 'C.', transform=ax.transAxes, fontsize=10, fontweight='bold')
ax.text(0.02, 0.9, 'Mooring D', transform=ax.transAxes, fontsize=10, fontweight='bold')
ax1 = ax.twinx()
line2, = ax1.plot(D_ctd_mid.sel(time=time_range).time,D_ctd_mid.sel(time=time_range).sigma0,zorder=10,color='tab:red',label='Potential Density')
ax1.set_ylabel('Potential Density \n (kg/m$^{3}$)',color='tab:red')
ax1.tick_params(axis='y',colors='tab:red')
lines = [line1, line2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='best')
ax.tick_params(axis='y', colors='tab:blue')
ax.set_ylim(0,0.0042)
ax.set_xlabel('Time (Year-Month)')
ax.tick_params(axis='x', labelrotation=45)
# Major ticks = every month
ax.xaxis.set_major_locator(mdates.MonthLocator())

# Format tick labels as abbreviated month names
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.set_xlabel('2018')

# plt.savefig('Moorings_Shear_N2_2018_trough.png', dpi=600)


Things to note: 

1. TNBD seeing a lot more shear and N2 variability that DITD and D 

2. Interesting peak end of April should be explored 

3. Peaks in september likely link with HSSW arrival and tidal effects from equinox 

In [ ]:
# 2-panel subplot for Ri PDFs 
fig = plt.figure(figsize=(9,4),constrained_layout=True)
plt.subplot(121)
plt.hist(TNBD_Ri,bins=np.arange(0,200,5),density=True)
plt.xlabel('Ri')
plt.ylabel('PDF')

plt.subplot(122)
plt.hist(DITD_Ri,bins=np.arange(0,200,5),density=True)
plt.xlabel('Ri')
plt.ylabel('PDF')


In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig = plt.figure(figsize=(10,4), constrained_layout=True)

# ----- Panel 1 -----
ax1 = plt.subplot(121)
ax1.hist(TNBD_Ri, bins=np.arange(0,200,5), density=True)
ax1.set_xlabel('Ri')
ax1.set_ylabel('Probability Density Function')
ax1.set_title('TNBD',weight='bold')

# Create inset
axins = inset_axes(ax1, width="40%", height="40%", loc='upper center')

axins.hist(TNBD_Ri, bins=np.arange(0,1,0.05), density=True)

# Zoom region
axins.set_xlim(0,1)
axins.set_ylim(0, None)

# Optional: cleaner ticks
axins.tick_params(labelsize=8)

# ----- Panel 2 -----
ax2 = plt.subplot(122)
ax2.hist(DITD_Ri, bins=np.arange(0,200,5), density=True)
ax2.set_xlabel('Ri')
# ax2.set_ylabel('P')
ax2.set_title('DITD',weight='bold')

# Create inset
axins = inset_axes(ax2, width="40%", height="40%", loc='upper center')

axins.hist(DITD_Ri, bins=np.arange(0,1,0.05), density=True)

# Zoom region
axins.set_xlim(0,1)
axins.set_ylim(0, None)

# Optional: cleaner ticks
axins.tick_params(labelsize=8)

# plt.savefig('Ri_PDF.png', dpi=600)

### HSSW Arrival Time and Cross Correlation Between Moorings 

In [ ]:
time_range = slice('2018-09-01','2018-10-01')

In [ ]:
# FIGURE FOR PAPER: Zoom into September and show sal, temp, and speed at each mooring 
# zoom into september time 
fig = plt.figure(figsize=(9,8),constrained_layout = True)
time_range = slice('2018-09-01','2018-10-01')
ax1 = plt.subplot(511)
plt.plot(TNBD_ctd_bot.sel(time=time_range).time,TNBD_ctd_bot.sel(time=time_range).sal,label='TNBD',color='tab:blue',lw=1)
plt.plot(DITD_ctd_bot.sel(time=time_range).time,DITD_ctd_bot.sel(time=time_range).sal,label='DITD',color='tab:orange',lw=1)
plt.plot(D_ctd_mid.sel(time=time_range).time,D_ctd_mid.sel(time=time_range).sal,label='D',color='tab:green',lw=1)
plt.xticks([])
# plt.legend()
plt.ylabel('Practical \n Salinity')
ax1.text(0.95, 0.89, 'A.', transform=ax1.transAxes, fontsize=10, fontweight='bold')

ax2 = plt.subplot(512)
plt.plot(TNBD_vel_bot.sel(time=time_range).time,TNBD_vel_bot.sel(time=time_range).speed,color='tab:blue',label='TNBD',lw=1)
plt.plot(DITD_vel_top.sel(time=time_range).time,DITD_vel_top.sel(time=time_range).speed,color='tab:orange',label='DITD',lw=1)
plt.plot(D_vel_bot.sel(time=time_range).time,D_vel_bot.sel(time=time_range).Mod_0,color='tab:green',label='D',lw=1)
plt.xticks([])
plt.legend(loc='upper left', fontsize=8)
plt.ylabel('Speed (cm/s)')
ax2.text(0.95, 0.89, 'B.', transform=ax2.transAxes, fontsize=10, fontweight='bold')


ax5 = plt.subplot(513)
resample = '12h'
ds1 = TNBD_vel_bot.sel(time=time_range)
q = plt.quiver(
    ds1.time.resample(time=resample).mean().values,np.zeros(len(ds1.time.resample(time=resample).mean())),
    ds1.u.resample(time=resample).mean(),
    ds1.v.resample(time=resample).mean(),
    scale=200, units='width',color='tab:blue',label='TNBD 1128 m'
)
plt.quiverkey(q, X=0.85, Y=0.1, U=10, label='10 cm/s', labelpos='E')
# plt.ylabel('Speed (m/s)')
plt.yticks([])
plt.legend(loc='lower left')
plt.xticks([])
ax5.text(0.95, 0.89, 'C.', transform=ax5.transAxes, fontsize=10, fontweight='bold')


ax5 = plt.subplot(514)
resample = '12h'
ds1 = DITD_vel_top.sel(time=time_range)
q = plt.quiver(
    ds1.time.resample(time=resample).mean().values,np.zeros(len(ds1.time.resample(time=resample).mean())),
    ds1.u.resample(time=resample).mean(),
    ds1.v.resample(time=resample).mean(),
    scale=200, units='width',color='tab:orange',label='DITD 1121 m'
)
plt.quiverkey(q, X=0.85, Y=0.1, U=10, label='10 cm/s', labelpos='E')
# plt.ylabel('Speed (m/s)')
plt.yticks([])
plt.legend(loc='lower left')
plt.xticks([])
ax5.text(0.95, 0.89, 'D.', transform=ax5.transAxes, fontsize=10, fontweight='bold')


ax5 = plt.subplot(515)
resample = '12h'
ds1 = D_vel_bot.sel(time=time_range)
q = plt.quiver(
    ds1.time.resample(time=resample).mean().values,np.zeros(len(ds1.time.resample(time=resample).mean())),
    ds1.u.resample(time=resample).mean(),
    ds1.v.resample(time=resample).mean(),
    scale=200, units='width',color='tab:green',label='D 1135 m'
)
plt.quiverkey(q, X=0.85, Y=0.1, U=10, label='10 cm/s', labelpos='E')
# plt.ylabel('Speed (m/s)')
plt.yticks([])
plt.legend(loc='lower left')
plt.xticks(rotation=45)
# plt.xlabel('Time (Year-Month)')
ax5.text(0.95, 0.89, 'E.', transform=ax5.transAxes, fontsize=10, fontweight='bold')

# plt.savefig('Sal_currents_2018_trough.png', dpi=600)


#### Cross-correlation: TNBD and DITD 

In [ ]:
# step 1. De-trend the data by removing the mean

# signal 1: DITD salinity 
s1 = DITD_ctd_bot.sel(time=time_range).sal - DITD_ctd_bot.sel(time=time_range).sal.mean()

# signal 2: TNBD salinity 
s2 = TNBD_ctd_bot.sel(time=time_range).sal - TNBD_ctd_bot.sel(time=time_range).sal.mean()
# step 2. compute cross correlation 
from scipy.signal import correlate 

corr = correlate(s2,s1,mode='full')
n = len(s1)
# calculate lags used based on length of timeseries 
lags = np.arange(-n+1, n)
# convert lag to minutes 
dt_mins = 30 #minutes
lag_mins = lags * dt_mins # lag in minutes 
# normalize correlation to be from -1 to 1 
corr = corr / (np.std(s1.values) * np.std(s2.values) * n)
# identify best lag (i.e. |max value of correlation|)
best_index = np.argmax(corr)
best_lag = lag_mins[best_index]

In [ ]:
best_lag

In [ ]:
np.max(corr)

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(lag_mins/1440, corr)
plt.axvline(best_lag/1440, color='tab:red',linestyle='dashed')
plt.xlabel("Lag (days)")
plt.ylabel("Correlation")

In [ ]:
# align time series 
fig = plt.figure(figsize=(12,4),constrained_layout = True)
# time_range = slice('2018-09-13','2018-09-29')
plt.subplot(111)
plt.plot(TNBD_ctd_bot.sel(time=time_range).time,TNBD_ctd_bot.sel(time=time_range).sal)
plt.plot(DITD_ctd_bot.sel(time=time_range).time - pd.Timedelta(minutes = np.absolute(best_lag)),DITD_ctd_bot.sel(time=time_range).sal)
# plt.plot(D_ctd_mid.sel(time=time_range).time,D_ctd_mid.sel(time=time_range).sal)

In [ ]:
TNBD_DITD_lag = lag_mins
TNBD_DITD_corr = corr
TNBD_DITD_best_lag = best_lag

#### Cross-correlation: TNBD and D 

In [ ]:
# step 1. De-trend the data by removing the mean

# signal 1: DITD salinity 
s1 = D_ctd_mid.sel(time=time_range).sal - D_ctd_mid.sel(time=time_range).sal.mean()

# signal 2: TNBD salinity 
s2 = TNBD_ctd_bot.sel(time=time_range).sal - TNBD_ctd_bot.sel(time=time_range).sal.mean()
# step 2. compute cross correlation 
from scipy.signal import correlate 

corr = correlate(s2,s1,mode='full')
n = len(s1)
# calculate lags used based on length of timeseries 
lags = np.arange(-n+1, n)
# convert lag to minutes 
dt_mins = 30 #minutes
lag_mins = lags * dt_mins # lag in minutes 
# normalize correlation to be from -1 to 1 
corr = corr / (np.std(s1.values) * np.std(s2.values) * n)
# identify best lag (i.e. |max value of correlation|)
best_index = np.argmax(corr)
best_lag = lag_mins[best_index]

In [ ]:
best_lag

In [ ]:
np.max(corr)

In [ ]:
TNBD_D_lag = lag_mins
TNBD_D_corr = corr
TNBD_D_best_lag = best_lag

#### Cross-Correlation DITD and D 

In [ ]:
# step 1. De-trend the data by removing the mean

# signal 1: DITD salinity 
s1 = D_ctd_mid.sel(time=time_range).sal - D_ctd_mid.sel(time=time_range).sal.mean()

# signal 2: TNBD salinity 
s2 = DITD_ctd_bot.sel(time=time_range).sal - DITD_ctd_bot.sel(time=time_range).sal.mean()
# step 2. compute cross correlation 
from scipy.signal import correlate 

corr = correlate(s2,s1,mode='full')
n = len(s1)
# calculate lags used based on length of timeseries 
lags = np.arange(-n+1, n)
# convert lag to minutes 
dt_mins = 30 #minutes
lag_mins = lags * dt_mins # lag in minutes 
# normalize correlation to be from -1 to 1 
corr = corr / (np.std(s1.values) * np.std(s2.values) * n)
# identify best lag (i.e. |max value of correlation|)
best_index = np.argmax(corr)
best_lag = lag_mins[best_index]

In [ ]:
np.max(corr)

In [ ]:
best_lag

In [ ]:
best_index

In [ ]:
np.max(corr)

In [ ]:
fig = plt.figure(figsize=(12,4))
plt.plot(lag_mins/1440, corr)
plt.axvline(best_lag/1440, color='tab:red',linestyle='dashed')
plt.xlabel("Lag (days)")
plt.ylabel("Correlation")

In [ ]:
# align time series 
fig = plt.figure(figsize=(12,4),constrained_layout = True)
# time_range = slice('2018-09-13','2018-09-29')
plt.subplot(111)
plt.plot(DITD_ctd_bot.sel(time=time_range).time,DITD_ctd_bot.sel(time=time_range).sal)
plt.plot(D_ctd_mid.sel(time=time_range).time + pd.Timedelta(minutes = np.absolute(best_lag)),D_ctd_mid.sel(time=time_range).sal)

In [ ]:
DITD_D_lag = lag_mins
DITD_D_corr = corr
DITD_D_best_lag = best_lag

In [ ]:
fig = plt.figure(figsize=(10,8),constrained_layout=True)

ax1 = plt.subplot(311)
plt.plot(TNBD_ctd_bot.sel(time=time_range).time,TNBD_ctd_bot.sel(time=time_range).sal,color='tab:blue',label='TNBD')
plt.plot(DITD_ctd_bot.sel(time=time_range).time,DITD_ctd_bot.sel(time=time_range).sal,color='tab:orange',label='DITD')
plt.ylabel("Practical Salinity")
plt.xticks(rotation=45)
# Format x-axis as month-day
ax1.text(0.95, 0.89, 'A.', transform=ax1.transAxes, fontsize=10, fontweight='bold')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
plt.legend()

ax2 = plt.subplot(312)
plt.plot(TNBD_DITD_lag/1440, TNBD_DITD_corr,color='k')
plt.axvline(TNBD_DITD_best_lag/1440, color='tab:red',linestyle='dashed')
plt.xlabel("Lag (days)")
plt.ylabel("Pearson Correlation \n Coefficent")
ax2.text(0.95, 0.89, 'B.', transform=ax2.transAxes, fontsize=10, fontweight='bold')


ax3 = plt.subplot(313)
plt.plot(TNBD_ctd_bot.sel(time=time_range).time,TNBD_ctd_bot.sel(time=time_range).sal,color='tab:blue',label='TNBD')
plt.plot(DITD_ctd_bot.sel(time=time_range).time - pd.Timedelta(minutes = np.absolute(TNBD_DITD_best_lag)),DITD_ctd_bot.sel(time=time_range).sal,color='tab:orange',label='DITD')
plt.ylabel("Practical Salinity")
plt.xticks(rotation=45)
# Format x-axis as month-day
ax3.text(0.95, 0.89, 'C.', transform=ax3.transAxes, fontsize=10, fontweight='bold')
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))

# plt.savefig('Correlation_TNBD_DITD.png', dpi=600)


In [ ]:
fig = plt.figure(figsize=(10,8),constrained_layout=True)

ax4 = plt.subplot(311)
plt.plot(TNBD_ctd_bot.sel(time=time_range).time,TNBD_ctd_bot.sel(time=time_range).sal,color='tab:blue',label='TNBD')
plt.plot(D_ctd_mid.sel(time=time_range).time,D_ctd_mid.sel(time=time_range).sal,color='tab:green',label='D')
plt.xticks(rotation=45)
plt.legend()
# Format x-axis as month-day
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
plt.ylabel("Practical Salinity")
ax4.text(0.95, 0.89, 'A.', transform=ax4.transAxes, fontsize=10, fontweight='bold')


ax3 = plt.subplot(312)
plt.plot(TNBD_D_lag/1440, TNBD_D_corr,color='k')
plt.axvline(TNBD_D_best_lag/1440, color='tab:red',linestyle='dashed')
# plt.xlabel("Lag (days)")
plt.ylabel("Pearson Correlation \n Coefficent")
ax3.text(0.95, 0.89, 'B.', transform=ax3.transAxes, fontsize=10, fontweight='bold')


ax4 = plt.subplot(313)
plt.plot(TNBD_ctd_bot.sel(time=time_range).time,TNBD_ctd_bot.sel(time=time_range).sal,color='tab:blue',label='TNBD')
plt.plot(D_ctd_mid.sel(time=time_range).time + pd.Timedelta(minutes = np.absolute(TNBD_D_best_lag)),D_ctd_mid.sel(time=time_range).sal,color='tab:green',label='D')
plt.xticks(rotation=45)
plt.legend()
# Format x-axis as month-day
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
plt.ylabel("Practical Salinity")
ax4.text(0.95, 0.89, 'C.', transform=ax4.transAxes, fontsize=10, fontweight='bold')

# plt.savefig('Correlation_TNBD_D.png', dpi=600)

In [ ]:
vel_shifted = D_vel_mid.shift(time = int(DITD_D_best_lag/30))

In [ ]:
fig = plt.figure(figsize=(12,4))
time_range_zoom = slice('2018-09-23','2018-09-26')
plt.plot(vel_shifted.sel(time=time_range_zoom).time,vel_shifted.sel(time=time_range_zoom).u,label='u-velocity')
plt.plot(vel_shifted.sel(time=time_range_zoom).time,vel_shifted.sel(time=time_range_zoom).v,label='v-velocity')


plt.axhline(y=0,color='k',linestyle='dashed',alpha=0.7)
plt.legend()

In [ ]:
fig = plt.figure(figsize=(12,4))
time_range_zoom = slice('2018-09-13','2018-09-17')
plt.plot(D_vel_mid.sel(time=time_range_zoom).time,D_vel_mid.sel(time=time_range_zoom).u,label='u-velocity')
plt.plot(D_vel_mid.sel(time=time_range_zoom).time,D_vel_mid.sel(time=time_range_zoom).v,label='v-velocity')


plt.axhline(y=0,color='k',linestyle='dashed',alpha=0.7)
plt.legend()

For cyclonic abyssal circulation to be true, what current directions do we expect to see at each mooring location? 

D: Northeast to connect to TNBD 

TNBD: North / northeast depending on extent of circulation, S/SW to connect to DITD, weaker speed becuase in center of trough 

DITD: North / North west 

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import mark_inset

In [ ]:
DITD_D_best_lag

In [ ]:
plt.close('all')

In [ ]:
fig = plt.figure(figsize=(9,12),constrained_layout=True)

ctd_shifted = D_ctd_mid.shift(time=int(DITD_D_best_lag/30))
vel_shifted = D_vel_mid.shift(time = int(DITD_D_best_lag/30))

ax6 = plt.subplot(611)
time_range = slice('2018-09-01','2018-10-01')
plt.plot(DITD_ctd_bot.sel(time=time_range).time,DITD_ctd_bot.sel(time=time_range).sal,color='tab:orange',label='DITD')
plt.plot(D_ctd_mid.sel(time=time_range).time ,D_ctd_mid.sel(time=time_range).sal,color='tab:green',label='D')
plt.xticks(rotation=45)
plt.legend()
# Format x-axis as month-day
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
plt.ylabel("Practical Salinity")
ax6.text(0.95, 0.89, 'A.', transform=ax6.transAxes, fontsize=10, fontweight='bold')
plt.ylim(34.79,34.86)


ax5 = plt.subplot(612)
plt.plot(DITD_D_lag/1440, DITD_D_corr,color='k')
plt.axvline(DITD_D_best_lag/1440, color='tab:red',linestyle='dashed')
plt.xlabel("Lag (days)")
plt.ylabel("Pearson Correlation \n Coefficent")
ax5.text(0.95, 0.89, 'B.', transform=ax5.transAxes, fontsize=10, fontweight='bold')
ax5.text(0.01, 0.89, 'Best Lag = +9.08 Days', transform=ax5.transAxes, fontsize=10, fontweight='bold')
plt.ylim(-1,1)
plt.axhline(y=0,color='k',linestyle='dashed',alpha=0.5)

# --- Panel 3: Zoomed-in salinity around peaks
ax_overview = plt.subplot(613)

# Crop to a few days around HSSW pea

# Plot salinity
ax_overview.plot(DITD_ctd_bot.sel(time=time_range).time, DITD_ctd_bot.sel(time=time_range).sal, color='tab:orange', label='DITD')
ax_overview.plot(ctd_shifted.sel(time=time_range).time, ctd_shifted.sel(time=time_range).sal, color='tab:green', label='D (lagged)')
# ax3.tick_params(axis='x', labelbottom=False)
plt.ylim(34.79,34.86)
ax_overview.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax_overview.tick_params('x',rotation=45)
zoom_start = pd.Timestamp('2018-09-22')
zoom_end = pd.Timestamp('2018-09-29')

ax_overview.axvspan(zoom_start, zoom_end,
                    color='grey',
                    alpha=0.15)
ax_overview.text(0.95, 0.89, 'C.', transform=ax_overview.transAxes, fontsize=10, fontweight='bold')
ax_overview.text(0.01, 0.89, 'Mooring D Shifted +9.08 Days', transform=ax_overview.transAxes, fontsize=10, fontweight='bold')
plt.ylabel('Practical Salinity')

# --- Panel 3: Zoomed-in salinity around peaks
ax_zoom = plt.subplot(614)


# Crop to a few days around HSSW peak
time_range_zoom = slice('2018-09-22','2018-09-28')
ditd_crop = DITD_ctd_bot.sel(time=time_range_zoom)
lagged_crop = ctd_shifted.sel(time=time_range_zoom)

# Plot salinity
ax_zoom.plot(ditd_crop.time, ditd_crop.sal, color='tab:orange', label='DITD')
ax_zoom.plot(lagged_crop.time, lagged_crop.sal, color='tab:green', label='D (lagged)')
ax_zoom.tick_params(axis='x', labelbottom=False)
plt.ylim(34.79,34.86)
ax_zoom.text(0.95, 0.89, 'D.', transform=ax_zoom.transAxes, fontsize=10, fontweight='bold')
plt.ylabel('Practical Salinity')

ax = plt.subplot(615)

plt.plot(vel_shifted.sel(time=time_range_zoom).time,vel_shifted.sel(time=time_range_zoom).u,label='u-velocity')
plt.plot(vel_shifted.sel(time=time_range_zoom).time,vel_shifted.sel(time=time_range_zoom).v,label='v-velocity',color='tab:red')

plt.axhline(y=0,color='k',linestyle='dashed',alpha=0.7)
ax.tick_params(axis='x', labelbottom=False)
# plt.legend()
plt.ylim(-27,27)
ax.text(0.95, 0.89, 'E.', transform=ax.transAxes, fontsize=10, fontweight='bold')
plt.ylabel('Velocity (cm/s)')
ax.text(0.01, 0.89, 'D', transform=ax.transAxes, fontsize=10, fontweight='bold')

ax = plt.subplot(616)
plt.plot(DITD_vel_top.sel(time=time_range_zoom).time,DITD_vel_top.sel(time=time_range_zoom).u,label='u-velocity')
plt.plot(DITD_vel_top.sel(time=time_range_zoom).time,DITD_vel_top.sel(time=time_range_zoom).v,label='v-velocity',color='tab:red')

plt.axhline(y=0,color='k',linestyle='dashed',alpha=0.7)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax.tick_params(axis='x', rotation=45)
plt.ylim(-27,27)
plt.legend()
ax.text(0.95, 0.89, 'F.', transform=ax.transAxes, fontsize=10, fontweight='bold')
plt.ylabel('Velocity (cm/s)')
ax.text(0.01, 0.89, 'DITD', transform=ax.transAxes, fontsize=10, fontweight='bold')

plt.suptitle('Cross Correlation: Mooring D and DITD',fontsize=12,fontweight='bold')
# plt.savefig('Correlation_DITD_D.png', dpi=600)

In [ ]:
TNBD_D_best_lag

In [ ]:
fig = plt.figure(figsize=(9,12),constrained_layout=True)

ctd_shifted = D_ctd_mid.shift(time=int(TNBD_D_best_lag/30))
vel_shifted = D_vel_mid.shift(time = int(TNBD_D_best_lag/30))

ax6 = plt.subplot(611)
time_range = slice('2018-09-01','2018-10-01')
plt.plot(TNBD_ctd_bot.sel(time=time_range).time,TNBD_ctd_bot.sel(time=time_range).sal,color='tab:blue',label='TNBD')
plt.plot(D_ctd_mid.sel(time=time_range).time,D_ctd_mid.sel(time=time_range).sal,color='tab:green',label='D')
plt.xticks(rotation=45)
plt.legend()
# Format x-axis as month-day
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
plt.ylabel("Practical Salinity")
ax6.text(0.95, 0.89, 'A.', transform=ax6.transAxes, fontsize=10, fontweight='bold')
plt.ylim(34.79,34.88)


ax5 = plt.subplot(612)
plt.plot(TNBD_D_lag/1440, TNBD_D_corr,color='k')
plt.axvline(TNBD_D_best_lag/1440, color='tab:red',linestyle='dashed')
plt.xlabel("Lag (days)")
plt.ylabel("Pearson Correlation \n Coefficent")
ax5.text(0.95, 0.89, 'B.', transform=ax5.transAxes, fontsize=10, fontweight='bold')
ax5.text(0.01, 0.89, 'Best Lag = +3.08 Days', transform=ax5.transAxes, fontsize=10, fontweight='bold')
plt.ylim(-1,1)
plt.axhline(y=0,color='k',linestyle='dashed',alpha=0.5)

# --- Panel 3: Zoomed-in salinity around peaks
ax_overview = plt.subplot(613)

# Crop to a few days around HSSW pea

# Plot salinity
ax_overview.plot(TNBD_ctd_bot.sel(time=time_range).time, TNBD_ctd_bot.sel(time=time_range).sal, color='tab:blue', label='TNBD')
ax_overview.plot(ctd_shifted.sel(time=time_range).time, ctd_shifted.sel(time=time_range).sal, color='tab:green', label='D (lagged)')
# ax3.tick_params(axis='x', labelbottom=False)
plt.ylim(34.79,34.88)
ax_overview.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax_overview.tick_params('x',rotation=45)
zoom_start = pd.Timestamp('2018-09-16')
zoom_end = pd.Timestamp('2018-09-23')

ax_overview.axvspan(zoom_start, zoom_end,
                    color='grey',
                    alpha=0.15)
ax_overview.text(0.95, 0.89, 'C.', transform=ax_overview.transAxes, fontsize=10, fontweight='bold')
ax_overview.text(0.01, 0.89, 'Mooring D Shifted +3.08 Days', transform=ax_overview.transAxes, fontsize=10, fontweight='bold')
plt.ylabel('Practical Salinity')

# --- Panel 3: Zoomed-in salinity around peaks
ax_zoom = plt.subplot(614)


# Crop to a few days around HSSW peak
time_range_zoom = slice('2018-09-16','2018-09-22')
tnbd_crop = TNBD_ctd_bot.sel(time=time_range_zoom)
lagged_crop = ctd_shifted.sel(time=time_range_zoom)

# Plot salinity
ax_zoom.plot(tnbd_crop.time, tnbd_crop.sal, color='tab:blue', label='TNBD')
ax_zoom.plot(lagged_crop.time, lagged_crop.sal, color='tab:green', label='D (lagged)')
ax_zoom.tick_params(axis='x', labelbottom=False)
plt.ylim(34.79,34.88)
ax_zoom.text(0.95, 0.89, 'D.', transform=ax_zoom.transAxes, fontsize=10, fontweight='bold')
plt.ylabel('Practical Salinity')

ax = plt.subplot(615)

plt.plot(vel_shifted.sel(time=time_range_zoom).time,vel_shifted.sel(time=time_range_zoom).u,label='u-velocity')
plt.plot(vel_shifted.sel(time=time_range_zoom).time,vel_shifted.sel(time=time_range_zoom).v,label='v-velocity',color='tab:red')

plt.axhline(y=0,color='k',linestyle='dashed',alpha=0.7)
ax.tick_params(axis='x', labelbottom=False)
# plt.legend()
plt.ylim(-27,27)
ax.text(0.95, 0.89, 'E.', transform=ax.transAxes, fontsize=10, fontweight='bold')
plt.ylabel('Velocity (cm/s)')
ax.text(0.01, 0.89, 'Mooring D', transform=ax.transAxes, fontsize=10, fontweight='bold')

ax = plt.subplot(616)
plt.plot(TNBD_vel_bot.sel(time=time_range_zoom).time,TNBD_vel_bot.sel(time=time_range_zoom).u,label='u-velocity')
plt.plot(TNBD_vel_bot.sel(time=time_range_zoom).time,TNBD_vel_bot.sel(time=time_range_zoom).v,label='v-velocity',color='tab:red')

plt.axhline(y=0,color='k',linestyle='dashed',alpha=0.7)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax.tick_params(axis='x', rotation=45)
plt.ylim(-27,27)
plt.legend()
ax.text(0.95, 0.89, 'F.', transform=ax.transAxes, fontsize=10, fontweight='bold')
ax.text(0.01, 0.89, 'TNBD', transform=ax.transAxes, fontsize=10, fontweight='bold')
plt.ylabel('Velocity (cm/s)')

plt.suptitle('Cross Correlation: Mooring D and TNBD',fontsize=12,fontweight='bold')
# plt.savefig('Correlation_D_TNBD.png', dpi=600)

In [ ]:
TNBD_DITD_best_lag

In [ ]:
fig = plt.figure(figsize=(9,12),constrained_layout=True)

ctd_shifted = DITD_ctd_bot.shift(time=int(TNBD_DITD_best_lag/30))
vel_shifted = DITD_vel_top.shift(time = int(TNBD_DITD_best_lag/30))

ax6 = plt.subplot(611)
time_range = slice('2018-09-01','2018-10-01')
plt.plot(TNBD_ctd_bot.sel(time=time_range).time,TNBD_ctd_bot.sel(time=time_range).sal,color='tab:blue',label='TNBD')
plt.plot(DITD_ctd_bot.sel(time=time_range).time,DITD_ctd_bot.sel(time=time_range).sal,color='tab:orange',label='DITD')
plt.xticks(rotation=45)
plt.legend()
# Format x-axis as month-day
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
plt.ylabel("Practical Salinity")
ax6.text(0.95, 0.89, 'A.', transform=ax6.transAxes, fontsize=10, fontweight='bold')
plt.ylim(34.79,34.88)


ax5 = plt.subplot(612)
plt.plot(TNBD_DITD_lag/1440, TNBD_DITD_corr,color='k')
plt.axvline(TNBD_DITD_best_lag/1440, color='tab:red',linestyle='dashed')
plt.xlabel("Lag (days)")
plt.ylabel("Pearson Correlation \n Coefficent")
ax5.text(0.95, 0.89, 'B.', transform=ax5.transAxes, fontsize=10, fontweight='bold')
ax5.text(0.01, 0.89, 'Best Lag = -6.18 Days', transform=ax5.transAxes, fontsize=10, fontweight='bold')
plt.ylim(-1,1)
plt.axhline(y=0,color='k',linestyle='dashed',alpha=0.5)

# --- Panel 3: Zoomed-in salinity around peaks
ax_overview = plt.subplot(613)

# Crop to a few days around HSSW pea

# Plot salinity
ax_overview.plot(TNBD_ctd_bot.sel(time=time_range).time, TNBD_ctd_bot.sel(time=time_range).sal, color='tab:blue', label='TNBD')
ax_overview.plot(ctd_shifted.sel(time=time_range).time, ctd_shifted.sel(time=time_range).sal, color='tab:orange', label='DITD (lagged)')
# ax3.tick_params(axis='x', labelbottom=False)
plt.ylim(34.79,34.88)
ax_overview.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax_overview.tick_params('x',rotation=45)
zoom_start = pd.Timestamp('2018-09-16')
zoom_end = pd.Timestamp('2018-09-23')

ax_overview.axvspan(zoom_start, zoom_end,
                    color='grey',
                    alpha=0.15)
ax_overview.text(0.95, 0.89, 'C.', transform=ax_overview.transAxes, fontsize=10, fontweight='bold')
ax_overview.text(0.01, 0.89, 'Mooring DITD Shifted -6.18 Days', transform=ax_overview.transAxes, fontsize=10, fontweight='bold')
plt.ylabel('Practical Salinity')

# --- Panel 3: Zoomed-in salinity around peaks
ax_zoom = plt.subplot(614)


# Crop to a few days around HSSW peak
time_range_zoom = slice('2018-09-16','2018-09-22')
tnbd_crop = TNBD_ctd_bot.sel(time=time_range_zoom)
lagged_crop = ctd_shifted.sel(time=time_range_zoom)

# Plot salinity
ax_zoom.plot(tnbd_crop.time, tnbd_crop.sal, color='tab:blue', label='TNBD')
ax_zoom.plot(lagged_crop.time, lagged_crop.sal, color='tab:orange', label='DITD (lagged)')
ax_zoom.tick_params(axis='x', labelbottom=False)
plt.ylim(34.79,34.88)
ax_zoom.text(0.95, 0.89, 'D.', transform=ax_zoom.transAxes, fontsize=10, fontweight='bold')
plt.ylabel('Practical Salinity')

ax = plt.subplot(615)

plt.plot(vel_shifted.sel(time=time_range_zoom).time,vel_shifted.sel(time=time_range_zoom).u,label='u-velocity')
plt.plot(vel_shifted.sel(time=time_range_zoom).time,vel_shifted.sel(time=time_range_zoom).v,label='v-velocity',color='tab:red')

plt.axhline(y=0,color='k',linestyle='dashed',alpha=0.7)
ax.tick_params(axis='x', labelbottom=False)
# plt.legend()
plt.ylim(-27,27)
ax.text(0.95, 0.89, 'E.', transform=ax.transAxes, fontsize=10, fontweight='bold')
plt.ylabel('Velocity (cm/s)')
ax.text(0.01, 0.89, 'DITD', transform=ax.transAxes, fontsize=10, fontweight='bold')

ax = plt.subplot(616)
plt.plot(TNBD_vel_bot.sel(time=time_range_zoom).time,TNBD_vel_bot.sel(time=time_range_zoom).u,label='u-velocity')
plt.plot(TNBD_vel_bot.sel(time=time_range_zoom).time,TNBD_vel_bot.sel(time=time_range_zoom).v,label='v-velocity',color='tab:red')

plt.axhline(y=0,color='k',linestyle='dashed',alpha=0.7)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax.tick_params(axis='x', rotation=45)
plt.ylim(-27,27)
plt.legend()
ax.text(0.95, 0.89, 'F.', transform=ax.transAxes, fontsize=10, fontweight='bold')
ax.text(0.01, 0.89, 'TNBD', transform=ax.transAxes, fontsize=10, fontweight='bold')
plt.ylabel('Velocity (cm/s)')

plt.suptitle('Cross Correlation: DITD and TNBD',fontsize=12,fontweight='bold')
# plt.savefig('Correlation_DITD_TNBD.png', dpi=600)

In [ ]:
fig = plt.figure(figsize=(10,8),constrained_layout=True)

ax1 = plt.subplot(3,2,1)
plt.plot(TNBD_DITD_lag/1440, TNBD_DITD_corr,color='k')
plt.axvline(TNBD_DITD_best_lag/1440, color='tab:red',linestyle='dashed')
# plt.xlabel("Lag (days)")
plt.ylabel("Pearson Correlation Coefficent")
ax1.text(0.95, 0.89, 'A.', transform=ax1.transAxes, fontsize=10, fontweight='bold')

ax2 = plt.subplot(3,2,2)
plt.plot(TNBD_ctd_bot.sel(time=time_range).time,TNBD_ctd_bot.sel(time=time_range).sal,color='tab:blue',label='TNBD')
plt.plot(DITD_ctd_bot.sel(time=time_range).time - pd.Timedelta(minutes = np.absolute(TNBD_DITD_best_lag)),DITD_ctd_bot.sel(time=time_range).sal,color='tab:orange',label='DITD')
plt.xticks(rotation=45)
plt.legend()
plt.ylabel("Practical Salinity")
# Format x-axis as month-day
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
ax2.text(0.95, 0.89, 'B.', transform=ax2.transAxes, fontsize=10, fontweight='bold')

ax3 = plt.subplot(3,2,3)
plt.plot(TNBD_D_lag/1440, TNBD_D_corr,color='k')
plt.axvline(TNBD_D_best_lag/1440, color='tab:red',linestyle='dashed')
# plt.xlabel("Lag (days)")
plt.ylabel("Pearson Correlation Coefficent")
ax3.text(0.95, 0.89, 'C.', transform=ax3.transAxes, fontsize=10, fontweight='bold')


ax4 = plt.subplot(3,2,4)
plt.plot(TNBD_ctd_bot.sel(time=time_range).time,TNBD_ctd_bot.sel(time=time_range).sal,color='tab:blue',label='TNBD')
plt.plot(D_ctd_mid.sel(time=time_range).time + pd.Timedelta(minutes = np.absolute(TNBD_D_best_lag)),D_ctd_mid.sel(time=time_range).sal,color='tab:green',label='D')
plt.xticks(rotation=45)
plt.legend()
# Format x-axis as month-day
ax4.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
plt.ylabel("Practical Salinity")
ax4.text(0.95, 0.89, 'D.', transform=ax4.transAxes, fontsize=10, fontweight='bold')

ax5 = plt.subplot(3,2,5)
plt.plot(DITD_D_lag/1440, DITD_D_corr,color='k')
plt.axvline(DITD_D_best_lag/1440, color='tab:red',linestyle='dashed')
plt.xlabel("Lag (days)")
plt.ylabel("Pearson Correlation Coefficent")
ax5.text(0.95, 0.89, 'E.', transform=ax5.transAxes, fontsize=10, fontweight='bold')

ax6 = plt.subplot(3,2,6)
plt.plot(DITD_ctd_bot.sel(time=time_range).time,DITD_ctd_bot.sel(time=time_range).sal,color='tab:orange',label='DITD')
plt.plot(D_ctd_mid.sel(time=time_range).time + pd.Timedelta(minutes = np.absolute(DITD_D_best_lag)),D_ctd_mid.sel(time=time_range).sal,color='tab:green',label='D')
plt.xticks(rotation=45)
plt.legend()
# Format x-axis as month-day
ax6.xaxis.set_major_formatter(mdates.DateFormatter('%b-%d'))
plt.ylabel("Practical Salinity")
ax6.text(0.95, 0.89, 'F.', transform=ax6.transAxes, fontsize=10, fontweight='bold')

# plt.savefig('Mooring_Cross_correlation_2018_trough.png',dpi=600)

In [ ]:
TNBD_Ri.where(TNBD_Ri < 0.25, drop=True)

In [ ]:
# Convert your Ri < 0.25 times to pandas Timestamps
low_Ri_times = pd.to_datetime([
    '2018-04-15T12:30:00.000000000', '2018-04-15T13:00:00.000000000',
       '2018-04-15T13:30:00.000000000', '2018-04-15T14:00:00.000000000',
       '2018-04-15T14:30:00.000000000', '2018-04-15T15:00:00.000000000',
       '2018-04-15T15:30:00.000000000', '2018-04-15T16:00:00.000000000',
       '2018-04-15T16:30:00.000000000', '2018-04-15T17:00:00.000000000',
       '2018-04-15T21:30:00.000000000', '2018-04-15T22:00:00.000000000',
       '2018-04-15T23:00:00.000000000', '2018-04-15T23:30:00.000000000',
       '2018-04-16T00:00:00.000000000', '2018-04-16T00:30:00.000000000',
       '2018-04-16T01:00:00.000000000', '2018-04-16T01:30:00.000000000',
       '2018-04-16T02:00:00.000000000', '2018-04-16T02:30:00.000000000',
       '2018-04-16T03:00:00.000000000', '2018-04-16T03:30:00.000000000',
       '2018-04-16T04:00:00.000000000', '2018-04-16T04:30:00.000000000',
       '2018-04-16T05:00:00.000000000', '2018-04-16T05:30:00.000000000',
       '2018-04-16T06:00:00.000000000', '2018-04-16T06:30:00.000000000',
       '2018-04-16T07:00:00.000000000', '2018-04-16T07:30:00.000000000',
       '2018-04-16T13:30:00.000000000', '2018-04-16T14:00:00.000000000',
       '2018-04-16T15:30:00.000000000', '2018-04-18T00:00:00.000000000',
       '2018-04-18T00:30:00.000000000', '2018-04-18T02:00:00.000000000',
       '2018-04-18T08:00:00.000000000', '2018-04-18T19:30:00.000000000',
       '2018-04-18T20:00:00.000000000', '2018-04-18T20:30:00.000000000',
       '2018-04-18T21:00:00.000000000', '2018-04-18T21:30:00.000000000',
       '2018-04-18T22:00:00.000000000', '2018-04-18T22:30:00.000000000',
       '2018-04-18T23:00:00.000000000', '2018-04-18T23:30:00.000000000',
       '2018-04-19T00:00:00.000000000', '2018-04-19T01:30:00.000000000',
       '2018-04-19T02:00:00.000000000', '2018-04-22T23:00:00.000000000',
       '2018-04-24T11:30:00.000000000', '2018-04-24T12:00:00.000000000',
       '2018-04-24T13:00:00.000000000', '2018-04-24T14:30:00.000000000',
       '2018-04-24T15:00:00.000000000', '2018-04-26T13:00:00.000000000',
       '2018-04-26T13:30:00.000000000', '2018-04-26T16:00:00.000000000',
       '2018-04-27T03:00:00.000000000', '2018-04-27T19:30:00.000000000',
       '2018-04-28T13:00:00.000000000', '2018-04-28T13:30:00.000000000',
       '2018-04-28T14:00:00.000000000', '2018-04-28T14:30:00.000000000',
       '2018-04-29T07:30:00.000000000', '2018-04-29T08:30:00.000000000',
       '2018-04-29T09:00:00.000000000', '2018-04-29T09:30:00.000000000',
       '2018-04-29T10:00:00.000000000', '2018-04-29T10:30:00.000000000',
       '2018-04-29T11:00:00.000000000', '2018-04-29T11:30:00.000000000',
       '2018-04-29T12:00:00.000000000', '2018-04-29T12:30:00.000000000',
       '2018-04-29T13:00:00.000000000', '2018-04-29T13:30:00.000000000',
       '2018-04-29T14:00:00.000000000', '2018-04-29T14:30:00.000000000',
       '2018-04-29T15:00:00.000000000', '2018-04-29T15:30:00.000000000',
       '2018-04-29T16:00:00.000000000', '2018-04-29T16:30:00.000000000',
       '2018-04-29T17:00:00.000000000', '2018-04-30T08:30:00.000000000',
       '2018-04-30T09:30:00.000000000', '2018-04-30T10:00:00.000000000',
       '2018-04-30T10:30:00.000000000', '2018-04-30T11:00:00.000000000',
       '2018-04-30T11:30:00.000000000', '2018-04-30T12:30:00.000000000',
       '2018-04-30T13:00:00.000000000', '2018-05-07T16:30:00.000000000',
       '2018-05-07T19:00:00.000000000', '2018-05-07T19:30:00.000000000',
       '2018-05-07T20:00:00.000000000', '2018-05-07T22:30:00.000000000',
       '2018-05-07T23:00:00.000000000', '2018-05-08T00:00:00.000000000',
       '2018-05-08T00:30:00.000000000', '2018-05-08T01:00:00.000000000',
       '2018-05-08T02:00:00.000000000', '2018-07-22T12:30:00.000000000',
       '2018-09-09T03:30:00.000000000', '2018-09-09T04:30:00.000000000',
       '2018-09-09T05:00:00.000000000', '2018-09-09T08:00:00.000000000',
       '2018-09-09T08:30:00.000000000', '2018-09-09T09:00:00.000000000',
       '2018-09-09T20:00:00.000000000', '2018-09-09T22:00:00.000000000',
       '2018-09-10T02:30:00.000000000', '2018-09-10T03:00:00.000000000',
       '2018-09-10T04:00:00.000000000', '2018-09-10T04:30:00.000000000',
       '2018-09-10T05:00:00.000000000', '2018-09-10T05:30:00.000000000',
       '2018-09-10T06:00:00.000000000', '2018-09-10T06:30:00.000000000',
       '2018-09-10T07:00:00.000000000', '2018-09-10T07:30:00.000000000',
       '2018-09-10T08:00:00.000000000', '2018-09-10T08:30:00.000000000',
       '2018-09-10T09:00:00.000000000', '2018-09-10T09:30:00.000000000',
       '2018-09-10T10:00:00.000000000', '2018-09-10T10:30:00.000000000',
       '2018-09-10T11:00:00.000000000', '2018-09-10T11:30:00.000000000',
       '2018-09-10T12:00:00.000000000', '2018-09-10T12:30:00.000000000',
       '2018-09-10T13:00:00.000000000', '2018-09-10T13:30:00.000000000',
       '2018-09-10T14:00:00.000000000', '2018-09-10T14:30:00.000000000',
       '2018-09-10T15:00:00.000000000', '2018-09-10T16:00:00.000000000',
       '2018-09-10T16:30:00.000000000', '2018-09-17T03:00:00.000000000',
       '2018-09-17T03:30:00.000000000', '2018-09-17T04:00:00.000000000',
       '2018-09-17T04:30:00.000000000', '2018-09-17T05:00:00.000000000',
       '2018-09-17T06:00:00.000000000'
])

# Width of the highlight band (±15 min)
band_width = pd.Timedelta(minutes=15)

In [ ]:
# plot the results 
fig = plt.figure(figsize=(11,6),constrained_layout=True)
time_range = slice('2018-03-14','2018-12-31')

ax = plt.subplot(211)
line1, = plt.plot(TNBD_vel_bot.sel(time=time_range).time,TNBD_vel_bot.sel(time=time_range).shear**2,label='Shear',zorder=1)
plt.ylabel('Shear$^{-2}$ (s$^{-2}$)',color='tab:blue')
# ax.text(0.95, 0.9, 'A.', transform=ax.transAxes, fontsize=10, fontweight='bold')
ax.text(0.02, 0.9, 'Mooring TNBD', transform=ax.transAxes, fontsize=10, fontweight='bold')
ax1 = ax.twinx()
line2, = ax1.plot(TNBD_ctd_bot.sel(time=time_range).time,TNBD_ctd_bot.sel(time=time_range).N2,zorder=10,color='tab:red',label='N$^2$')
ax1.set_ylabel('N$^2$ (s$^{-2}$)',color='tab:red')
ax.set_ylim(0,1.7e-5)
ax1.set_ylim(0,8.9e-6)
ax.tick_params(axis='x', labelrotation=45)
ax1.tick_params(axis='y',colors='tab:red')
ax.tick_params(axis='y', colors='tab:blue')
lines = [line1, line2]
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='best')
ax.tick_params(axis='x', labelbottom=False)

# Highlight low Ri times
for t in low_Ri_times:
    ax1.axvspan(t - band_width, t + band_width, color='yellow', alpha=0.2, zorder=0)

ax = plt.subplot(212)
plt.plot(DITD_vel_bot.sel(time=time_range).time,DITD_vel_bot.sel(time=time_range).shear**2,label='Shear',zorder=1)
plt.ylabel('Shear$^{-2}$ (s$^{-2}$)',color='tab:blue')
# ax.text(0.95, 0.9, 'B.', transform=ax.transAxes, fontsize=10, fontweight='bold')
ax.text(0.02, 0.9, 'Mooring DITD', transform=ax.transAxes, fontsize=10, fontweight='bold')
ax1 = ax.twinx()
ax1.plot(DITD_ctd_bot.sel(time=time_range).time,DITD_ctd_bot.sel(time=time_range).N2,zorder=10,color='tab:red')
ax1.set_ylabel('N$^2$ (s$^{-2}$)',color='tab:red')
ax.set_ylim(0,1.7e-5)
ax1.set_ylim(0,8.9e-6)
# ax.tick_params(axis='x', labelrotation=45)
ax1.tick_params(axis='y',colors='tab:red')
ax.tick_params(axis='y', colors='tab:blue')
# ax.tick_params(axis='x', labelbottom=False)
ax.set_xlabel('Time (Year-Month)')
ax.tick_params(axis='x', labelrotation=45)
# Major ticks = every month
ax.xaxis.set_major_locator(mdates.MonthLocator())

# Format tick labels as abbreviated month names
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.set_xlabel('2018')

# ax = plt.subplot(313)
# line1, = plt.plot(D_vel_bot.sel(time=time_range).time,D_vel_bot.sel(time=time_range).shear,label='Shear',zorder=1)
# plt.ylabel('Shear (s$^{-1}$)',color='tab:blue')
# # ax.text(0.95, 0.9, 'C.', transform=ax.transAxes, fontsize=10, fontweight='bold')
# ax.text(0.02, 0.9, 'Mooring D', transform=ax.transAxes, fontsize=10, fontweight='bold')
# ax1 = ax.twinx()
# line2, = ax1.plot(D_ctd_mid.sel(time=time_range).time,D_ctd_mid.sel(time=time_range).sigma0,zorder=10,color='tab:red',label='Potential Density')
# ax1.set_ylabel('Potential Density \n (kg/m$^{3}$)',color='tab:red')
# ax1.tick_params(axis='y',colors='tab:red')
# lines = [line1, line2]
# labels = [l.get_label() for l in lines]
# ax1.legend(lines, labels, loc='best')
# ax.tick_params(axis='y', colors='tab:blue')
# ax.set_ylim(0,0.0042)
# ax.set_xlabel('Time (Year-Month)')
# ax.tick_params(axis='x', labelrotation=45)
# # Major ticks = every month
# ax.xaxis.set_major_locator(mdates.MonthLocator())

# # Format tick labels as abbreviated month names
# ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
# ax.set_xlabel('2018')

# plt.savefig('Moorings_Shear_N2_2018_trough.png', dpi=600)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.dates as mdates

time_range= slice('2018-09-01','2018-10-01')
# Convert your Ri < 0.25 times to pandas Timestamps
low_Ri_times = pd.to_datetime([
    '2018-09-09T03:30:00', '2018-09-09T04:30:00', '2018-09-09T05:00:00',
    '2018-09-09T08:00:00', '2018-09-09T08:30:00', '2018-09-09T09:00:00',
    '2018-09-09T20:00:00', '2018-09-09T22:00:00',
    '2018-09-10T02:30:00', '2018-09-10T03:00:00',
    '2018-09-10T04:00:00', '2018-09-10T04:30:00',
    '2018-09-10T05:00:00', '2018-09-10T05:30:00',
    '2018-09-10T06:00:00', '2018-09-10T06:30:00',
    '2018-09-10T07:00:00', '2018-09-10T07:30:00',
    '2018-09-10T08:00:00', '2018-09-10T08:30:00',
    '2018-09-10T09:00:00', '2018-09-10T09:30:00',
    '2018-09-10T10:00:00', '2018-09-10T10:30:00',
    '2018-09-10T11:00:00', '2018-09-10T11:30:00',
    '2018-09-10T12:00:00', '2018-09-10T12:30:00',
    '2018-09-10T13:00:00', '2018-09-10T13:30:00',
    '2018-09-10T14:00:00', '2018-09-10T14:30:00',
    '2018-09-10T15:00:00', '2018-09-10T16:00:00',
    '2018-09-10T16:30:00', '2018-09-17T03:00:00',
    '2018-09-17T03:30:00', '2018-09-17T04:00:00',
    '2018-09-17T04:30:00', '2018-09-17T05:00:00',
    '2018-09-17T06:00:00'
])

# Width of the highlight band (±15 min)
band_width = pd.Timedelta(minutes=15)

# Create figure
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10,6), sharex=True, constrained_layout=True)

# --- Top subplot: Salinity & Temperature ---
ax1.plot(TNBD_ctd_top.sel(time=time_range).time,
         TNBD_ctd_top.sel(time=time_range).sal,
         color='tab:purple', label='Salinity')

ax1.set_ylabel('Practical Salinity')
ax1.set_ylim(34.79,34.84)
ax1.legend(loc='upper left')
# ax1.set_title('Mooring TNBD - Sept/Oct 2018')

# Twin axis for temperature
ax1b = ax1.twinx()
ax1b.plot(TNBD_ctd_top.sel(time=time_range).time,
          TNBD_ctd_top.sel(time=time_range).theta,
          color='tab:green', label='Temperature')
ax1b.set_ylabel('Potential \n Temperature (°C)')
ax1b.legend(loc='upper right')

# Highlight low Ri times
for t in low_Ri_times:
    ax1.axvspan(t - band_width, t + band_width, color='grey', alpha=0.2, zorder=0)


# --- Top subplot: Salinity & Temperature ---
ax2.plot(TNBD_ctd_bot.sel(time=time_range).time,
         TNBD_ctd_bot.sel(time=time_range).sal,
         color='tab:purple', label='Salinity')

ax2.set_ylabel('Practical Salinity')
ax2.set_ylim(34.81,34.87)
# ax2.legend(loc='upper left')
# ax2.set_title('Mooring TNBD - Sept/Oct 2018')

# Twin axis for temperature
ax2b = ax2.twinx()
ax2b.plot(TNBD_ctd_bot.sel(time=time_range).time,
          TNBD_ctd_bot.sel(time=time_range).theta,
          color='tab:green', label='Temperature')
ax2b.set_ylabel('Potential \n Temperature (°C)')
# ax2b.legend(loc='upper right')

# Highlight low Ri times
for t in low_Ri_times:
    ax2.axvspan(t - band_width, t + band_width, color='grey', alpha=0.2, zorder=0)

# --- Bottom subplot: Shear & N² ---
ax3.plot(TNBD_vel_bot.sel(time=time_range).time,
         TNBD_vel_bot.sel(time=time_range).shear**2,
         color='tab:blue', label='Shear', zorder=1)
ax3.set_ylabel('Shear (s$^{-1}$)', color='tab:blue')
ax3.tick_params(axis='y', colors='tab:blue')
# ax3.set_ylim(0,0.0042)

# Twin axis for N²
ax3b = ax3.twinx()
ax3b.plot(TNBD_ctd_bot.sel(time=time_range).time,
          TNBD_ctd_bot.sel(time=time_range).N2,
          color='tab:red', label='N²', zorder=10)
ax3b.set_ylabel('N² (s$^{-2}$)', color='tab:red')
ax3b.tick_params(axis='y', colors='tab:red')

# Highlight low Ri times
for t in low_Ri_times:
    ax3.axvspan(t - band_width, t + band_width, color='grey', alpha=0.2, zorder=0)

# --- Format x-axis ---
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax3.set_xlabel('(Month-Day) 2018')
fig.autofmt_xdate()

plt.show()
plt.savefig('TNBD_shear_N2_sept.png', dpi=600)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.dates as mdates

time_range= slice('2018-04-01','2018-09-11')
# Convert your Ri < 0.25 times to pandas Timestamps
low_Ri_times = pd.to_datetime([
    '2018-09-09T03:30:00', '2018-09-09T04:30:00', '2018-09-09T05:00:00',
    '2018-09-09T08:00:00', '2018-09-09T08:30:00', '2018-09-09T09:00:00',
    '2018-09-09T20:00:00', '2018-09-09T22:00:00',
    '2018-09-10T02:30:00', '2018-09-10T03:00:00',
    '2018-09-10T04:00:00', '2018-09-10T04:30:00',
    '2018-09-10T05:00:00', '2018-09-10T05:30:00',
    '2018-09-10T06:00:00', '2018-09-10T06:30:00',
    '2018-09-10T07:00:00', '2018-09-10T07:30:00',
    '2018-09-10T08:00:00', '2018-09-10T08:30:00',
    '2018-09-10T09:00:00', '2018-09-10T09:30:00',
    '2018-09-10T10:00:00', '2018-09-10T10:30:00',
    '2018-09-10T11:00:00', '2018-09-10T11:30:00',
    '2018-09-10T12:00:00', '2018-09-10T12:30:00',
    '2018-09-10T13:00:00', '2018-09-10T13:30:00',
    '2018-09-10T14:00:00', '2018-09-10T14:30:00',
    '2018-09-10T15:00:00', '2018-09-10T16:00:00',
     '2018-09-10T16:30:00'])
# '2018-09-17T03:00:00',
#     '2018-09-17T03:30:00', '2018-09-17T04:00:00',
#     '2018-09-17T04:30:00', '2018-09-17T05:00:00',
#     '2018-09-17T06:00:00'
# ])

# Width of the highlight band (±15 min)
band_width = pd.Timedelta(minutes=15)

# Create figure
fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10,9), sharex=True, constrained_layout=True)

# --- Top subplot: Salinity & Temperature ---
ax1.plot(TNBD_ctd_top.sel(time=time_range).time,
         TNBD_ctd_top.sel(time=time_range).sal,
         color='tab:purple', label='Salinity')

ax1.set_ylabel('Practical Salinity',color='tab:purple')
ax1.tick_params(axis='y', colors='tab:purple')
# ax1.set_ylim(34.79,34.84)
ax1.legend(loc='upper left')
# ax1.set_title('Mooring TNBD - Sept/Oct 2018')

# Twin axis for temperature
ax1b = ax1.twinx()
ax1b.plot(TNBD_ctd_top.sel(time=time_range).time,
          TNBD_ctd_top.sel(time=time_range).theta,
          color='tab:green', label='Temperature')
ax1b.set_ylabel('Potential \n Temperature (°C)',color='tab:green')
ax1b.tick_params(axis='y', colors='tab:green')
ax1b.legend(loc='upper right')
ax1.text(0.14, 0.89, '(A) TNBD 1031m', transform=ax1.transAxes, fontsize=10, fontweight='bold')

# Highlight low Ri times
for t in low_Ri_times:
    ax1.axvspan(t - band_width, t + band_width, color='grey', alpha=0.2, zorder=0)


# --- Top subplot: Salinity & Temperature ---
ax2.plot(TNBD_ctd_bot.sel(time=time_range).time,
         TNBD_ctd_bot.sel(time=time_range).sal,
         color='tab:purple', label='Salinity')

ax2.set_ylabel('Practical Salinity',color='tab:purple')
ax2.tick_params(axis='y', colors='tab:purple')
# ax2.set_ylim(34.81,34.87)
# ax2.legend(loc='upper left')
# ax2.set_title('Mooring TNBD - Sept/Oct 2018')

# Twin axis for temperature
ax2b = ax2.twinx()
ax2b.plot(TNBD_ctd_bot.sel(time=time_range).time,
          TNBD_ctd_bot.sel(time=time_range).theta,
          color='tab:green', label='Temperature')
ax2b.set_ylabel('Potential \n Temperature (°C)',color='tab:green')
ax2b.tick_params(axis='y', colors='tab:green')
ax2.text(0.8, 0.89, '(B) TNBD 1128m', transform=ax2.transAxes, fontsize=10, fontweight='bold')
# ax2b.legend(loc='upper right')

# Highlight low Ri times
for t in low_Ri_times:
    ax2.axvspan(t - band_width, t + band_width, color='grey', alpha=0.2, zorder=0)







# --- Top subplot: Salinity & Temperature ---
ax3.plot(TNBD_vel_top.sel(time=time_range).time,
         TNBD_vel_top.sel(time=time_range).u,
         color='tab:blue', label='u')

ax3.plot(TNBD_vel_top.sel(time=time_range).time,
         TNBD_vel_top.sel(time=time_range).v,
         color='tab:red', label='v')

ax3.set_ylabel('Velocity (cm/s)')
ax3.set_ylim(-43,43)
# ax3.set_ylim(34.79,34.84)
ax3.legend(loc='upper left')
# ax1.set_title('Mooring TNBD - Sept/Oct 2018')

# Twin axis for Speed
ax3b = ax3.twinx()
ax3b.plot(TNBD_vel_top.sel(time=time_range).time,
          TNBD_vel_top.sel(time=time_range).speed,
          color='k', label='Speed')
ax3b.set_ylabel('Speed (cm/s)')
ax3b.legend(loc='upper right')
ax3b.set_ylim(0,43)
ax3.text(0.1, 0.89, '(C) TNBD 1031m', transform=ax3.transAxes, fontsize=10, fontweight='bold')

# Highlight low Ri times
for t in low_Ri_times:
    ax3.axvspan(t - band_width, t + band_width, color='grey', alpha=0.2, zorder=0)






ax4.plot(TNBD_vel_bot.sel(time=time_range).time,
         TNBD_vel_bot.sel(time=time_range).u,
         color='tab:blue', label='u')

ax4.plot(TNBD_vel_bot.sel(time=time_range).time,
         TNBD_vel_bot.sel(time=time_range).v,
         color='tab:red', label='v')

ax4.set_ylabel('Velocity (cm/s)')
ax4.set_ylim(-43,43)
ax4.legend(loc='upper left')
# ax1.set_title('Mooring TNBD - Sept/Oct 2018')

# Twin axis for Speed
ax4b = ax4.twinx()
ax4b.plot(TNBD_vel_bot.sel(time=time_range).time,
          TNBD_vel_bot.sel(time=time_range).speed,
          color='k', label='Speed')
ax4b.set_ylabel('Speed (cm/s)')
ax4b.set_ylim(0,43)
ax4b.legend(loc='upper right')
ax4.text(0.1, 0.89, '(D) TNBD 1128m', transform=ax4.transAxes, fontsize=10, fontweight='bold')

# Highlight low Ri times
for t in low_Ri_times:
    ax4.axvspan(t - band_width, t + band_width, color='grey', alpha=0.2, zorder=0)



# --- Bottom subplot: Shear & N² ---
ax5.plot(TNBD_vel_bot.sel(time=time_range).time,
         TNBD_vel_bot.sel(time=time_range).shear,
         color='tab:blue', label='Shear', zorder=1)
ax5.set_ylabel('Shear (s$^{-1}$)', color='tab:blue')
ax5.tick_params(axis='y', colors='tab:blue')
ax5.set_ylim(0,0.0042)

# Twin axis for N²
ax5b = ax5.twinx()
ax5b.plot(TNBD_ctd_bot.sel(time=time_range).time,
          TNBD_ctd_bot.sel(time=time_range).N2,
          color='tab:red', label='N²', zorder=10)
ax5b.set_ylabel('N² (s$^{-2}$)', color='tab:red')
ax5b.tick_params(axis='y', colors='tab:red')
# ax5b.set_ylim(0,8.5e-6)

# Highlight low Ri times
for t in low_Ri_times:
    ax5.axvspan(t - band_width, t + band_width, color='grey', alpha=0.2, zorder=0)

# --- Format x-axis ---
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
ax5.set_xlabel('(Month-Day) 2018')
fig.autofmt_xdate()

plt.show()
# plt.savefig('TNBD_shear_N2_sept_zoomed2.png', dpi=600)


In [ ]:
fig = plt.figure()
plt.scatter(DITD_vel_bot.sel(time=DITD_vel_bot.time).shear,DITD_vel_top.sel(time=DITD_vel_bot.time).speed,label='1121m',color='tab:blue')
plt.scatter(DITD_vel_bot.shear,DITD_vel_bot.speed,label='1218m',color='tab:orange')
plt.legend(loc='lower right')
# plt.scatter(TNBD_vel_bot.sel(time=time_range).shear,TNBD_vel_top.sel(time=time_range).speed)
plt.xlabel('Shear (s$^{-1}$)')
plt.ylabel('Speed (cm/s)')
plt.title('DITD March-June 2018')
plt.xlim(-0.0001,0.0035)
plt.ylim(-0.0001,29)

In [ ]:
fig = plt.figure()
plt.scatter(TNBD_vel_bot.sel(time=slice('2018-03','2018-06')).shear,TNBD_vel_top.sel(time=slice('2018-03','2018-06')).speed,label='1031m')
plt.scatter(TNBD_vel_bot.sel(time=slice('2018-03','2018-06')).shear,TNBD_vel_bot.sel(time=slice('2018-03','2018-06')).speed,label='1128m')
plt.legend(loc='lower right')
# plt.scatter(TNBD_vel_bot.sel(time=time_range).shear,TNBD_vel_top.sel(time=time_range).speed)
plt.xlabel('Shear (s$^{-1}$)')
plt.ylabel('Speed (cm/s)')
plt.title('TNBD March-June 2018')
plt.xlim(-0.0001,0.0035)
plt.ylim(-0.0001,29)

In [ ]:
fig = plt.figure()
plt.scatter(TNBD_vel_bot.sel(time=time_range).shear,TNBD_vel_bot.sel(time=time_range).speed)
# plt.scatter(TNBD_vel_bot.sel(time=time_range).shear,TNBD_vel_top.sel(time=time_range).speed)
plt.xlabel('Shear (s$^{-1}$)')
plt.ylabel('Speed (cm/s) Bottom Sensor')